WHO > [The Global Health Observatory: Explore a world of health data](https://www.who.int/data/gho/data/themes/topics/indicator-groups/indicator-group-details/GHO/life-expectancy-and-healthy-life-expectancy)<br />
WHO > [Life expectancy at birth](https://www.who.int/data/gho/data/indicators/indicator-details/GHO/life-expectancy-at-birth-(years))<br />
WHO > [Healthy life expectancy (HALE) at birth](https://www.who.int/data/gho/data/indicators/indicator-details/GHO/gho-ghe-hale-healthy-life-expectancy-at-birth)<br />

Lists of countries by life expectancy: [world](https://en.wikipedia.org/wiki/List_of_countries_by_life_expectancy) <i>([template](https://en.wikipedia.org/wiki/Template:List_of_countries_by_life_expectancy/World_Health_Organization))</i>, [Europe](https://en.wikipedia.org/wiki/List_of_European_countries_by_life_expectancy), [Asia](https://en.wikipedia.org/wiki/List_of_Asian_countries_by_life_expectancy), [North America](https://en.wikipedia.org/wiki/List_of_North_American_countries_by_life_expectancy), [South America](https://en.wikipedia.org/wiki/List_of_South_American_countries_by_life_expectancy), [Americas (combined)](https://en.wikipedia.org/wiki/List_of_countries_in_the_Americas_by_life_expectancy), [Africa](https://en.wikipedia.org/wiki/List_of_African_countries_by_life_expectancy), [regions](https://en.wikipedia.org/wiki/List_of_world_regions_by_life_expectancy), [EU](https://en.wikipedia.org/wiki/Demographics_of_the_European_Union#Life_expectancy)<br />
[my private wiki-page for exploration](https://en.wikipedia.org/wiki/User:Lady3mlnm/HALE)<br>
Списки стран по ожидаемой продолжительности жизни: [мир в целом](https://ru.wikipedia.org/wiki/Список_стран_по_ожидаемой_продолжительности_жизни), [Европа](https://ru.wikipedia.org/wiki/Список_стран_Европы_по_ожидаемой_продолжительности_жизни), [Азия](https://ru.wikipedia.org/wiki/Список_стран_Азии_по_ожидаемой_продолжительности_жизни), [Северная Америка](https://ru.wikipedia.org/wiki/Список_стран_Южной_Америки_по_ожидаемой_продолжительности_жизни), [Северная Америка](https://ru.wikipedia.org/wiki/Список_стран_Южной_Америки_по_ожидаемой_продолжительности_жизни),  [Африка](https://ru.wikipedia.org/wiki/Список_стран_Африки_по_ожидаемой_продолжительности_жизни), [регионы](https://ru.wikipedia.org/wiki/Список_регионов_мира_по_ожидаемой_продолжительности_жизни), [ЕС](https://ru.wikipedia.org/wiki/Население_стран_Европейского_союза#Продолжительность_жизни), [СНГ](https://ru.wikipedia.org/wiki/Содружество_Независимых_Государств#Социальное_развитие_стран_СНГ)<br />
[Список государств и зависимых территорий по населению](https://ru.wikipedia.org/wiki/Список_государств_и_зависимых_территорий_по_населению)

In [2]:
import pandas as pd
import math
import re

In [3]:
pd.options.display.max_rows = 100
pd.options.display.min_rows = 6
pd.options.display.max_columns = 50
pd.options.display.float_format = '{:.2f}'.format

DESTINATION_OUTPUT = 'file'  # to where table code should be placed: 'file', 'write_here' or, otherwise, just print 'Done'
YEAR_SELECTED = 2019

In [4]:
# list of countries that should be ignored during processing (it is needed when WBG made correction for only some countries.
# So processing of countries without correction is redundant.
LS_IGNORE = \
    ['occupied Palestinian territory']

LS_COUNTRIES_EXAMPLES = ['Russia', 'USA', 'France', 'Spain', 'Japan', 'Switzerland', 'South Korea', 'World']

In [5]:
LS_WHO_REGIONS = ['Africa', 'Americas', 'Eastern Mediterranean', 'Europe', 'South-East Asia', 'Western Pacific']
LS_INCOME_GROUPS = ['High-income', 'Upper-middle-income', 'Lower-middle-income', 'Low-income']

In [6]:
def output_table_code(st, file_name='', destination=''):
    if not destination:
        destination = DESTINATION_OUTPUT
        
    if destination == 'file':
        with open('output-tables/'+file_name, 'w', encoding="utf-8") as fh:
            fh.write(st)
        print('Data has written to file')
    elif destination == 'write_here':
        print(st)
    else:
        print('done')

In [7]:
dd_countries_renaming = {
	'Bolivia (Plurinational State of)': 'Bolivia',
	'Brunei Darussalam': 'Brunei',
	'Cabo Verde': 'Cape Verde',
	"Democratic People's Republic of Korea": 'North Korea',
	'Democratic Republic of the Congo': 'Congo DR',
    'Congo': 'Congo Republic',
	'Iran (Islamic Republic of)': 'Iran',
	"Lao People's Democratic Republic": 'Laos',
	'Micronesia (Federated States of)': 'Micronesia',
	'Netherlands (Kingdom of the)': 'Netherlands',
	'Republic of Korea': 'South Korea',
	'Republic of Moldova': 'Moldova',
	'Russian Federation': 'Russia',
	'Syrian Arab Republic': 'Syria',
    'Türkiye': 'Turkey',
	'United Kingdom of Great Britain and Northern Ireland': 'United Kingdom',
	'United Republic of Tanzania': 'Tanzania',
	'United States of America': 'USA',
	'Venezuela (Bolivarian Republic of)': 'Venezuela',
    'Viet Nam': 'Vietnam',
    'Saint Vincent and the Grenadines': 'St. Vincent and the Grenadines',
	'occupied Palestinian territory, including east Jerusalem': 'occupied Palestinian territory',
    'Global': 'World'
}

In [8]:
def load_data_from_csv(file_name_core, selected_indicator, selected_years, prefix_for_cols):

    def load_single_csv(file_name):
        return pd.read_csv(f"data/{file_name}", usecols=['Indicator', 'Location', 'Period', 'Dim1', 'FactValueNumeric'])

    def get_pretty_df_for_sex(df, selected_sex, prefix_for_cols_with_sex):
        df_sex = df[df.sex == selected_sex] \
                   .drop(columns='sex') \
                   .pivot(index='country', columns='year', values='value')
        df_sex.columns = [f'{prefix_for_cols_with_sex}_{df_sex.columns[0]}', f'{prefix_for_cols_with_sex}']
        df_sex.index.name = ''
        return df_sex


    df = pd.concat([load_single_csv(f"{file_name_core} -countries.csv"),
                    load_single_csv(f"{file_name_core} -global.csv"),
                    load_single_csv(f"{file_name_core} -regions.csv"),
                    load_single_csv(f"{file_name_core} -income_groups.csv")])
    
    
    df.rename(columns={'Location': 'country',
                       'Period': 'year',
                       'Dim1': 'sex',
                       'FactValueNumeric': 'value'}, inplace=True)
    
    df = df[(df.Indicator == selected_indicator) & (df.year.isin(selected_years))] \
              .drop(columns='Indicator')
            
    df['country'] = df['country'].replace(dd_countries_renaming)
    
    return pd.concat([get_pretty_df_for_sex(df, 'Both sexes', f'{prefix_for_cols}_o'),
                      get_pretty_df_for_sex(df, 'Male', f'{prefix_for_cols}_m'),
                      get_pretty_df_for_sex(df, 'Female', f'{prefix_for_cols}_f')],
                          axis='columns')



df_le_birth = load_data_from_csv(file_name_core="Life expectancy at birth",
                                 selected_indicator="Life expectancy at birth (years)",
                                 selected_years=[2000, 2019], prefix_for_cols='LE_birth')
df_le_birth.loc[LS_COUNTRIES_EXAMPLES].sort_values(by='LE_birth_o', ascending=False)

,LE_birth_o_2000,LE_birth_o,LE_birth_m_2000,LE_birth_m,LE_birth_f_2000,LE_birth_f
,,,,,,
Japan,81.53,84.47,78.08,81.70,84.80,87.15
South Korea,75.86,83.69,72.24,80.56,79.35,86.56
Switzerland,79.71,83.48,76.90,81.76,82.34,85.12
Spain,79.09,83.14,75.69,80.54,82.48,85.66
France,78.91,82.53,75.24,79.77,82.52,85.15
USA,76.66,78.74,74.16,76.53,79.06,80.98
Russia,65.16,73.22,58.83,68.17,72.14,78.00
World,66.77,73.12,64.39,70.61,69.22,75.70


In [9]:
df_le_60 = load_data_from_csv(file_name_core="Life expectancy at birth",
                              selected_indicator="Life expectancy at age 60 (years)",
                              selected_years=[2000, 2019], prefix_for_cols='LE_60')
df_le_60.loc[LS_COUNTRIES_EXAMPLES].sort_values(by='LE_60_o', ascending=False)

,LE_60_o_2000,LE_60_o,LE_60_m_2000,LE_60_m,LE_60_f_2000,LE_60_f
,,,,,,
Japan,24.63,26.66,21.82,24.28,27.06,28.86
South Korea,20.33,26.13,17.90,23.55,22.23,28.30
France,23.01,25.69,20.39,23.63,25.29,27.51
Switzerland,22.99,25.45,20.84,24.09,24.80,26.68
Spain,22.70,25.39,20.34,23.28,24.82,27.33
USA,21.34,23.24,19.80,21.99,22.65,24.41
World,18.87,21.03,17.23,19.41,20.36,22.54
Russia,16.39,19.93,13.29,16.80,18.71,22.19


In [10]:
df_hale_birth = load_data_from_csv(file_name_core="Healthy life expectancy (HALE) at birth",
                                   selected_indicator="Healthy life expectancy (HALE) at birth (years)",
                                   selected_years=[2000, 2019], prefix_for_cols='HALE_birth')
df_hale_birth.loc[LS_COUNTRIES_EXAMPLES].sort_values(by='HALE_birth_o', ascending=False)

,HALE_birth_o_2000,HALE_birth_o,HALE_birth_m_2000,HALE_birth_m,HALE_birth_f_2000,HALE_birth_f
,,,,,,
Japan,71.11,73.58,68.96,72.09,73.11,75.01
South Korea,66.55,72.50,64.04,70.62,68.96,74.19
Spain,68.89,71.69,67.05,70.90,70.70,72.42
Switzerland,68.35,71.53,67.04,71.33,69.54,71.67
France,67.98,70.72,66.21,69.78,69.70,71.60
USA,65.32,66.02,64.19,65.13,66.38,66.93
Russia,56.71,63.72,51.90,60.41,62.01,66.84
World,58.12,63.45,56.96,62.33,59.32,64.59


In [11]:
df_hale_60 = load_data_from_csv(file_name_core="Healthy life expectancy (HALE) at birth",
                                selected_indicator="Healthy life expectancy (HALE) at age 60 (years)",
                                selected_years=[2000, 2019], prefix_for_cols='HALE_60')
df_hale_60.loc[LS_COUNTRIES_EXAMPLES].sort_values(by='HALE_60_o', ascending=False)

,HALE_60_o_2000,HALE_60_o,HALE_60_m_2000,HALE_60_m,HALE_60_f_2000,HALE_60_f
,,,,,,
Japan,18.85,20.42,16.78,18.79,20.65,21.95
South Korea,15.53,19.66,13.72,17.88,16.96,21.16
France,17.37,19.26,15.69,18.07,18.83,20.33
Spain,17.40,19.18,15.83,17.94,18.82,20.33
Switzerland,17.38,19.12,16.03,18.46,18.52,19.73
USA,15.67,16.59,14.70,15.82,16.50,17.31
World,14.28,15.80,13.27,14.87,15.20,16.67
Russia,12.28,14.92,9.99,12.75,13.99,16.49


In [12]:
df = pd.concat([df_le_birth['LE_birth_o'],
                df_le_birth['LE_birth_m'],
                df_le_birth['LE_birth_f'],
                (df_le_birth['LE_birth_f'] - df_le_birth['LE_birth_m']).round(2),
                (df_le_birth['LE_birth_o'] - df_le_birth['LE_birth_o_2000']).round(2),

                df_hale_birth['HALE_birth_o'],
                df_hale_birth['HALE_birth_m'],
                df_hale_birth['HALE_birth_f'],
                (df_hale_birth['HALE_birth_f'] - df_hale_birth['HALE_birth_m']).round(2),
                (df_hale_birth['HALE_birth_o'] - df_hale_birth['HALE_birth_o_2000']).round(2),

                100 * df_hale_birth['HALE_birth_o_2000'] / df_le_birth['LE_birth_o_2000'],
                100 * df_hale_birth['HALE_birth_o'] / df_le_birth['LE_birth_o'],

                df_le_60['LE_60_o'],
                df_le_60['LE_60_m'],
                df_le_60['LE_60_f'],
                (df_le_60['LE_60_f'] - df_le_60['LE_60_m']).round(2),
                (df_le_60['LE_60_o'] - df_le_60['LE_60_o_2000']).round(2),

                df_hale_60['HALE_60_o'],
                df_hale_60['HALE_60_m'],
                df_hale_60['HALE_60_f'],
                (df_hale_60['HALE_60_f'] - df_hale_60['HALE_60_m']).round(2),
                (df_hale_60['HALE_60_o'] - df_hale_60['HALE_60_o_2000']).round(2),

                100 * df_hale_60['HALE_60_o_2000'] / df_le_60['LE_60_o_2000'],
                100 * df_hale_60['HALE_60_o'] / df_le_60['LE_60_o'],
               ], axis='columns') \
                   .rename(columns = {0 : 'LE_birth_fΔm',
                                      1 : 'LE_birth_Δ_2000',
                                      2 : 'HALE_birth_fΔm',
                                      3 : 'HALE_birth_Δ_2000',
                                      4 : 'ratio_birth_2000',
                                      5 : 'ratio_birth',
                                      6 : 'LE_60_fΔm',
                                      7 : 'LE_60_Δ_2000',
                                      8 : 'HALE_60_fΔm',
                                      9 : 'HALE_60_Δ_2000',
                                      10: 'ratio_60_2000',
                                      11: 'ratio_60',
                                     }) \
                   .sort_values(by=['LE_birth_o', 'HALE_birth_o'], ascending=False)

df.insert(loc=11, value=df['ratio_birth']-df['ratio_birth_2000'], column='ratio_birth_Δ')
df.insert(loc=24, value=df['ratio_60']-df['ratio_60_2000'], column='ratio_60_Δ')

df.loc[LS_COUNTRIES_EXAMPLES].sort_values(by=['LE_birth_o', 'HALE_birth_o'], ascending=False)

,LE_birth_o,LE_birth_m,LE_birth_f,LE_birth_fΔm,LE_birth_Δ_2000,HALE_birth_o,HALE_birth_m,HALE_birth_f,HALE_birth_fΔm,HALE_birth_Δ_2000,ratio_birth_2000,ratio_birth_Δ,ratio_birth,LE_60_o,LE_60_m,LE_60_f,LE_60_fΔm,LE_60_Δ_2000,HALE_60_o,HALE_60_m,HALE_60_f,HALE_60_fΔm,HALE_60_Δ_2000,ratio_60_2000,ratio_60_Δ,ratio_60
,,,,,,,,,,,,,,,,,,,,,,,,,,
Japan,84.47,81.70,87.15,5.45,2.94,73.58,72.09,75.01,2.92,2.47,87.22,-0.11,87.11,26.66,24.28,28.86,4.58,2.03,20.42,18.79,21.95,3.16,1.57,76.53,0.06,76.59
South Korea,83.69,80.56,86.56,6.00,7.83,72.50,70.62,74.19,3.57,5.95,87.73,-1.10,86.63,26.13,23.55,28.30,4.75,5.80,19.66,17.88,21.16,3.28,4.13,76.39,-1.15,75.24
Switzerland,83.48,81.76,85.12,3.36,3.77,71.53,71.33,71.67,0.34,3.18,85.75,-0.06,85.69,25.45,24.09,26.68,2.59,2.46,19.12,18.46,19.73,1.27,1.74,75.60,-0.47,75.13
Spain,83.14,80.54,85.66,5.12,4.05,71.69,70.90,72.42,1.52,2.80,87.10,-0.88,86.23,25.39,23.28,27.33,4.05,2.69,19.18,17.94,20.33,2.39,1.78,76.65,-1.11,75.54
France,82.53,79.77,85.15,5.38,3.62,70.72,69.78,71.60,1.82,2.74,86.15,-0.46,85.69,25.69,23.63,27.51,3.88,2.68,19.26,18.07,20.33,2.26,1.89,75.49,-0.52,74.97
USA,78.74,76.53,80.98,4.45,2.08,66.02,65.13,66.93,1.80,0.70,85.21,-1.36,83.85,23.24,21.99,24.41,2.42,1.90,16.59,15.82,17.31,1.49,0.92,73.43,-2.04,71.39
Russia,73.22,68.17,78.00,9.83,8.06,63.72,60.41,66.84,6.43,7.01,87.03,-0.01,87.03,19.93,16.80,22.19,5.39,3.54,14.92,12.75,16.49,3.74,2.64,74.92,-0.06,74.86
World,73.12,70.61,75.70,5.09,6.35,63.45,62.33,64.59,2.26,5.33,87.05,-0.27,86.78,21.03,19.41,22.54,3.13,2.16,15.80,14.87,16.67,1.80,1.52,75.68,-0.54,75.13


<br>
<br>

In [14]:
# create code for placing info about countries in Wikipedia (for countries)
def create_table_countries(df, lang='en', file_header=None):

    def if_value(x, prec=2):
        return '—' if math.isnan(x) else \
               f"{x:0.{prec}f}"  if x>=0 else \
               f"−{-x:0.{prec}f}"

    if lang=='ru':
        file_header = file_header if file_header else 'who_stats_header_countries_ru.txt'
        ptn_1 = 'флагификация'
        ptn_2 = 'флаг'
        prettify_name = {
            "World": "\'\'\'Мир\'\'\'",
            "Europe": "\'\'\'Европа\'\'\'<ref>{{cite web|title=ВОЗ: Европа |publisher=Всемирная организация здравоохранения |url=https://www.who.int/europe/ru/ |access-date=2025-10-12}}</ref>",
            "Eastern Mediterranean": "\'\'\'Восточное Средиземноморье\'\'\'<ref>{{cite web|title=WHO: Eastern Mediterranean: Countries |lang=en |publisher=Всемирная организация здравоохранения |url=http://www.emro.who.int/countries.html |access-date=2025-10-12}}</ref>",
            "South-East Asia": "\'\'\'[[Юго-Восточная Азия]]\'\'\'<ref>{{cite web|title=WHO: South-East Asia: Where we work |lang=en |publisher=Всемирная организация здравоохранения |url=https://www.who.int/southeastasia/about/where-we-work |access-date=2025-10-12}}</ref>",
            "Western Pacific": "\'\'\'Западно-тихоокеанский регион\'\'\'<ref>{{cite web|title=WHO: Western Pacific: Where we work |lang=en |publisher=Всемирная организация здравоохранения |url=https://www.who.int/westernpacific/about/where-we-work |access-date=2025-10-12}}</ref>",
            "Americas": "\'\'\'[[Америка]]\'\'\'<ref>{{cite web|title=WHO: PAHO: Countries and Centers |lang=en |publisher=Всемирная организация здравоохранения |url=https://www.paho.org/en/countries-and-centers |access-date=2025-10-12}}</ref>",
            "Africa": "\'\'\'Африка\'\'\'<ref>{{cite web|title=WHO: Africa: Countries |lang=en |publisher=Всемирная организация здравоохранения |url=https://www.afro.who.int/countries |access-date=2025-10-12}}</ref>", 
        }
    else:
        file_header = file_header if file_header else 'who_stats_header_countries_en.txt'
        ptn_1 = 'flaglist'
        ptn_2 = 'flagicon'
        prettify_name = {
            "World": "\'\'\'World\'\'\'",
            "Europe": "\'\'\'Europe\'\'\'<ref>{{cite web|title=WHO: Europe |publisher=World Health Organization |url=https://www.who.int/europe |access-date=12 November 2025}}</ref>",
            "Eastern Mediterranean": "\'\'\'[[Eastern Mediterranean]]\'\'\'<ref>{{cite web|title=WHO: Eastern Mediterranean: Countries |publisher=World Health Organization |url=http://www.emro.who.int/countries.html |access-date=12 November 2025}}</ref>",
            "South-East Asia": "\'\'\'[[South-East Asia]]\'\'\'<ref>{{cite web|title=WHO: South-East Asia: Where we work |language=en |publisher=World Health Organization |url=https://www.who.int/southeastasia/about/where-we-work |access-date=12 November 2025}}</ref>",
            "Western Pacific": "\'\'\'Western Pacific\'\'\'<ref>{{cite web|title=WHO: Western Pacific: Where we work |publisher=World Health Organization |url=https://www.who.int/westernpacific/about/where-we-work |access-date=12 November 2025}}</ref>",
            "Americas": "\'\'\'[[Americas]]\'\'\'<ref>{{cite web|title=WHO: PAHO: Countries and Centers |publisher=World Health Organization |url=https://www.paho.org/en/countries-and-centers |access-date=12 November 2025}}</ref>",
            "Africa": "\'\'\'Africa\'\'\'<ref>{{cite web|title=WHO: Africa: Countries |language=en |publisher=World Health Organization |url=https://www.afro.who.int/countries |access-date=12 November 2025}}</ref>",
        }
            

    with open('design/' + file_header, mode='r', encoding="utf-8") as fh:
        st_header = fh.read()
        
    st_header = st_header.strip()
    st = ''
        
    for i in range(len(df)):
        ser = df.iloc[i]
        
        comp_LE_birth = ser.LE_birth_Δ_2000
        comp_HALE_birth = ser.HALE_birth_Δ_2000
        comp_LE_60 = ser.LE_60_Δ_2000
        comp_HALE_60 = ser.HALE_60_Δ_2000
        
        if ser.name in ['World', 'Europe', 'Eastern Mediterranean', 'South-East Asia', 'Western Pacific', 'Americas', 'Africa']:
            st += '\n' + '|-class=static-row-header\n' + \
                  f'|style="text-align:center"| {prettify_name.get(ser.name, ser.name)} ' + \
                  f'||style="background:#e0ffd8;"| \'\'\'{ser.LE_birth_o:0.2f}\'\'\' ' + \
                  f'||style="background:#eaf3ff;"| \'\'\'{ser.LE_birth_m:0.2f}\'\'\' ' + \
                  f'||style="background:#fee7f6;"| \'\'\'{ser.LE_birth_f:0.2f}\'\'\' ' + \
                  f'|| \'\'\'{if_value(ser.LE_birth_fΔm, 2)}\'\'\' ' + \
                  f'||style="background:#fff8dc;"| \'\'\'{if_value(comp_LE_birth, 2)}\'\'\' ' + \
                  f'||style="background:#e0ffd8; border-left-width:2px;"| \'\'\'{ser.HALE_birth_o:0.2f}\'\'\' ' + \
                  f'||style="background:#eaf3ff;"| \'\'\'{ser.HALE_birth_m:0.2f}\'\'\' ' + \
                  f'||style="background:#fee7f6;"| \'\'\'{ser.HALE_birth_f:0.2f}\'\'\' ' + \
                  f'|| \'\'\'{if_value(ser.HALE_birth_fΔm, 2)}\'\'\' ' + \
                  f'||style="background:#fff8dc;"| \'\'\'{if_value(comp_HALE_birth, 2)}\'\'\' ' + \
                  f'||style="background:#e0ffd8; border-left-width:3px;"| \'\'\'{ser.LE_60_o:0.2f}\'\'\' ' + \
                  f'||style="background:#eaf3ff;"| \'\'\'{ser.LE_60_m:0.2f}\'\'\' ' + \
                  f'||style="background:#fee7f6;"| \'\'\'{ser.LE_60_f:0.2f}\'\'\' ' + \
                  f'|| \'\'\'{if_value(ser.LE_60_fΔm, 2)}\'\'\' ' + \
                  f'||style="background:#fff8dc;"| \'\'\'{if_value(comp_LE_60, 2)}\'\'\' ' + \
                  f'||style="background:#e0ffd8; border-left-width:2px;"| \'\'\'{ser.HALE_60_o:0.2f}\'\'\' ' + \
                  f'||style="background:#eaf3ff;"| \'\'\'{ser.HALE_60_m:0.2f}\'\'\' ' + \
                  f'||style="background:#fee7f6;"| \'\'\'{ser.HALE_60_f:0.2f}\'\'\' ' + \
                  f'|| \'\'\'{if_value(ser.HALE_60_fΔm, 2)}\'\'\' ' + \
                  f'||style="background:#fff8dc;"| \'\'\'{if_value(comp_HALE_60, 2)}\'\'\' ' + \
                  f'||'
        else:
            st += '\n' + '|-\n' + \
                  f'| {{{{{ptn_1}|{ser.name}}}}} ' + \
                  f'||style="background:#e0ffd8;"| {ser.LE_birth_o:0.2f} ' + \
                  f'||style="background:#eaf3ff;"| {ser.LE_birth_m:0.2f} ' + \
                  f'||style="background:#fee7f6;"| {ser.LE_birth_f:0.2f} ' + \
                  f'|| {if_value(ser.LE_birth_fΔm, 2)} ' + \
                  f'||style="background:#fff8dc;| {if_value(comp_LE_birth, 2)} ' + \
                  f'||style="background:#e0ffd8;border-left-width:2px;"| {ser.HALE_birth_o:0.2f} ' + \
                  f'||style="background:#eaf3ff;"| {ser.HALE_birth_m:0.2f} ' + \
                  f'||style="background:#fee7f6;"| {ser.HALE_birth_f:0.2f} ' + \
                  f'|| {if_value(ser.HALE_birth_fΔm, 2)} ' + \
                  f'||style="background:#fff8dc;| {if_value(comp_HALE_birth, 2)} ' + \
                  f'||style="background:#e0ffd8;border-left-width:3px;"| {ser.LE_60_o:0.2f} ' + \
                  f'||style="background:#eaf3ff;"| {ser.LE_60_m:0.2f} ' + \
                  f'||style="background:#fee7f6;"| {ser.LE_60_f:0.2f} ' + \
                  f'|| {if_value(ser.LE_60_fΔm, 2)} ' + \
                  f'||style="background:#fff8dc;| {if_value(comp_LE_60, 2)} ' + \
                  f'||style="background:#e0ffd8;border-left-width:2px;"| {ser.HALE_60_o:0.2f} ' + \
                  f'||style="background:#eaf3ff;"| {ser.HALE_60_m:0.2f} ' + \
                  f'||style="background:#fee7f6;"| {ser.HALE_60_f:0.2f} ' + \
                  f'|| {if_value(ser.HALE_60_fΔm, 2)} ' + \
                  f'||style="background:#fff8dc;| {if_value(comp_HALE_60, 2)} ' + \
                  f'|| {{{{{ptn_2}|{ser.name}}}}}'
    st += '\n|}'
    
    # WARNING: be sure that inline styles do not contains numbers with comma
    if lang == 'ru':
        st = re.sub('(?<=\\d)\\.(?=\\d)', ',', st)  # replace . to comma, if this . is between two digits
        
    st = st_header + st

    return st

In [15]:
df_all_countries = df.drop(LS_WHO_REGIONS + LS_INCOME_GROUPS + LS_IGNORE)

print("Number of records for all countries:", len(df_all_countries))

df_all_countries

Number of records for all countries: 185


,LE_birth_o,LE_birth_m,LE_birth_f,LE_birth_fΔm,LE_birth_Δ_2000,HALE_birth_o,HALE_birth_m,HALE_birth_f,HALE_birth_fΔm,HALE_birth_Δ_2000,ratio_birth_2000,ratio_birth_Δ,ratio_birth,LE_60_o,LE_60_m,LE_60_f,LE_60_fΔm,LE_60_Δ_2000,HALE_60_o,HALE_60_m,HALE_60_f,HALE_60_fΔm,HALE_60_Δ_2000,ratio_60_2000,ratio_60_Δ,ratio_60
,,,,,,,,,,,,,,,,,,,,,,,,,,
Japan,84.47,81.70,87.15,5.45,2.94,73.58,72.09,75.01,2.92,2.47,87.22,-0.11,87.11,26.66,24.28,28.86,4.58,2.03,20.42,18.79,21.95,3.16,1.57,76.53,0.06,76.59
Singapore,83.90,81.81,85.98,4.17,5.38,73.80,72.66,74.89,2.23,4.39,88.40,-0.44,87.96,25.83,24.05,27.51,3.46,4.35,20.21,18.98,21.38,2.40,3.41,78.21,0.03,78.24
South Korea,83.69,80.56,86.56,6.00,7.83,72.50,70.62,74.19,3.57,5.95,87.73,-1.10,86.63,26.13,23.55,28.30,4.75,5.80,19.66,17.88,21.16,3.28,4.13,76.39,-1.15,75.24
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Somalia,55.21,52.87,57.75,4.88,5.93,48.65,47.28,50.14,2.86,5.24,88.09,0.03,88.12,13.52,12.42,14.59,2.17,1.17,10.41,9.72,11.08,1.36,0.88,77.17,-0.17,77.00
Central African Republic,52.93,50.31,55.89,5.58,8.70,45.97,44.41,47.75,3.34,7.67,86.59,0.26,86.85,12.61,11.19,14.27,3.08,1.48,9.42,8.44,10.56,2.12,1.15,74.30,0.40,74.70
Lesotho,51.78,48.96,54.91,5.95,4.40,45.04,43.39,46.88,3.49,3.22,88.27,-1.28,86.98,12.71,10.80,14.33,3.53,-0.41,9.52,8.24,10.59,2.35,-0.61,77.21,-2.31,74.90


In [16]:
table_code_all = create_table_countries(df_all_countries, lang='en', file_header='who_stats_header_countries_en_inFrame.txt')
output_table_code(table_code_all, 'Table code WHO -all_countries -en.txt', destination=DESTINATION_OUTPUT)

Data has written to file


In [17]:
table_code_all = create_table_countries(df_all_countries, lang='ru')
output_table_code(table_code_all, 'Table code WHO -all_countries -ru.txt', destination=DESTINATION_OUTPUT)

Data has written to file


<br />
<br />

In [19]:
ls_Europe = ['Albania', 'Armenia', 'Austria', 'Azerbaijan', 'Belarus', 'Belgium', 'Bosnia and Herzegovina', 'Bulgaria', 'Croatia',
             'Cyprus', 'Czechia', 'Denmark', 'Estonia', 'Finland', 'France', 'Georgia', 'Germany', 'Greece', 'Hungary', 'Iceland',
             'Ireland', 'Italy', 'Kazakhstan', 'Latvia', 'Lithuania', 'Luxembourg', 'Malta', 'Moldova', 'Montenegro', 'Netherlands',
             'North Macedonia', 'Norway', 'Poland', 'Portugal', 'Romania', 'Russia', 'Serbia', 'Slovakia', 'Slovenia', 'Spain',
             'Sweden', 'Switzerland', 'Turkey', 'Ukraine', 'United Kingdom',
             'World', 'Europe']

# small countries: Andorra, Liechtenstein, Monaco, San Marino, Vatican City
# main countries: France, Germany, Italy, Russia, Spain, United Kingdom

print("Number of records for Europe:", len(ls_Europe))

df_Europe = df.loc[ls_Europe]   \
              .sort_values(by=['LE_birth_o', 'HALE_birth_o', 'LE_60_o', 'HALE_60_o'], ascending=False)
df_Europe

Number of records for Europe: 47


,LE_birth_o,LE_birth_m,LE_birth_f,LE_birth_fΔm,LE_birth_Δ_2000,HALE_birth_o,HALE_birth_m,HALE_birth_f,HALE_birth_fΔm,HALE_birth_Δ_2000,ratio_birth_2000,ratio_birth_Δ,ratio_birth,LE_60_o,LE_60_m,LE_60_f,LE_60_fΔm,LE_60_Δ_2000,HALE_60_o,HALE_60_m,HALE_60_f,HALE_60_fΔm,HALE_60_Δ_2000,ratio_60_2000,ratio_60_Δ,ratio_60
,,,,,,,,,,,,,,,,,,,,,,,,,,
Switzerland,83.48,81.76,85.12,3.36,3.77,71.53,71.33,71.67,0.34,3.18,85.75,-0.06,85.69,25.45,24.09,26.68,2.59,2.46,19.12,18.46,19.73,1.27,1.74,75.60,-0.47,75.13
Spain,83.14,80.54,85.66,5.12,4.05,71.69,70.90,72.42,1.52,2.80,87.10,-0.88,86.23,25.39,23.28,27.33,4.05,2.69,19.18,17.94,20.33,2.39,1.78,76.65,-1.11,75.54
Italy,82.99,80.94,84.91,3.97,3.62,71.43,70.85,71.92,1.07,3.04,86.17,-0.10,86.07,25.14,23.53,26.56,3.03,2.60,18.95,18.03,19.78,1.75,1.94,75.47,-0.09,75.38
Luxembourg,82.80,80.83,84.74,3.91,4.38,71.46,70.98,71.89,0.91,3.61,86.52,-0.22,86.30,24.73,23.13,26.22,3.09,2.85,18.83,17.97,19.64,1.67,2.10,76.46,-0.32,76.14
Sweden,82.71,81.17,84.24,3.07,3.14,71.41,71.33,71.46,0.13,2.15,87.04,-0.71,86.34,24.78,23.57,25.91,2.34,2.36,18.80,18.26,19.32,1.06,1.57,76.85,-0.98,75.87
Norway,82.66,81.12,84.16,3.04,4.12,71.18,71.04,71.28,0.24,3.35,86.36,-0.25,86.11,24.72,23.57,25.78,2.21,2.75,18.66,18.09,19.20,1.11,2.02,75.74,-0.25,75.49
France,82.53,79.77,85.15,5.38,3.62,70.72,69.78,71.60,1.82,2.74,86.15,-0.46,85.69,25.69,23.63,27.51,3.88,2.68,19.26,18.07,20.33,2.26,1.89,75.49,-0.52,74.97
Iceland,82.47,81.10,83.90,2.80,2.78,71.39,71.30,71.46,0.16,2.25,86.76,-0.20,86.56,24.82,23.96,25.67,1.71,2.20,19.07,18.68,19.45,0.77,1.70,76.79,0.04,76.83
Netherlands,82.34,80.85,83.76,2.91,4.33,71.08,71.05,71.07,0.02,2.98,87.30,-0.97,86.32,24.65,23.43,25.76,2.33,3.33,18.64,18.07,19.16,1.09,2.16,77.30,-1.68,75.62


In [20]:
table_code_Europe = create_table_countries(df_Europe, lang='en')
output_table_code(table_code_Europe, 'Table code WHO -Europe -en.txt', destination=DESTINATION_OUTPUT)

Data has written to file


In [21]:
table_code_Europe = create_table_countries(df_Europe, lang='ru')
output_table_code(table_code_Europe, 'Table code WHO -Europe -ru.txt', destination=DESTINATION_OUTPUT)

Data has written to file


<br />
<br />

In [23]:
ls_Asia = ['Afghanistan', 'Armenia', 'Azerbaijan', 'Bahrain', 'Bangladesh', 'Bhutan', 'Brunei', 'Cambodia', 'China', 'Cyprus', 'Egypt',
           'Georgia', 'India', 'Indonesia', 'Iran', 'Iraq', 'Israel', 'Japan', 'Jordan', 'Kazakhstan', 'Kuwait', 'Kyrgyzstan', 'Laos',
           'Lebanon', 'Malaysia', 'Maldives', 'Mongolia', 'Myanmar', 'Nepal', 'North Korea', 'Oman', 'Pakistan', 'Philippines', 'Qatar',
           'Russia', 'Saudi Arabia', 'Singapore', 'South Korea', 'Sri Lanka', 'Syria', 'Tajikistan', 'Thailand', 'Timor-Leste', 'Turkey',
           'Turkmenistan', 'United Arab Emirates', 'Uzbekistan', 'Vietnam', 'Yemen',
           'World', 'Western Pacific', 'South-East Asia', 'Eastern Mediterranean']

# main countries: China, India, Indonesia, Israel, Japan, South Korea, Pakistan, Russia   # Bangladesh, Saudi Arabia

print("Number of records for Asia:", len(ls_Asia))

df_Asia = df.loc[ls_Asia]   \
            .sort_values(by=['LE_birth_o', 'HALE_birth_o', 'LE_60_o', 'HALE_60_o'], ascending=False)
df_Asia

Number of records for Asia: 53


,LE_birth_o,LE_birth_m,LE_birth_f,LE_birth_fΔm,LE_birth_Δ_2000,HALE_birth_o,HALE_birth_m,HALE_birth_f,HALE_birth_fΔm,HALE_birth_Δ_2000,ratio_birth_2000,ratio_birth_Δ,ratio_birth,LE_60_o,LE_60_m,LE_60_f,LE_60_fΔm,LE_60_Δ_2000,HALE_60_o,HALE_60_m,HALE_60_f,HALE_60_fΔm,HALE_60_Δ_2000,ratio_60_2000,ratio_60_Δ,ratio_60
,,,,,,,,,,,,,,,,,,,,,,,,,,
Japan,84.47,81.70,87.15,5.45,2.94,73.58,72.09,75.01,2.92,2.47,87.22,-0.11,87.11,26.66,24.28,28.86,4.58,2.03,20.42,18.79,21.95,3.16,1.57,76.53,0.06,76.59
Singapore,83.90,81.81,85.98,4.17,5.38,73.80,72.66,74.89,2.23,4.39,88.40,-0.44,87.96,25.83,24.05,27.51,3.46,4.35,20.21,18.98,21.38,2.40,3.41,78.21,0.03,78.24
South Korea,83.69,80.56,86.56,6.00,7.83,72.50,70.62,74.19,3.57,5.95,87.73,-1.10,86.63,26.13,23.55,28.30,4.75,5.80,19.66,17.88,21.16,3.28,4.13,76.39,-1.15,75.24
Israel,82.57,80.74,84.32,3.58,4.03,71.51,71.17,71.78,0.61,3.17,87.01,-0.41,86.61,24.88,23.61,26.02,2.41,2.93,18.91,18.29,19.47,1.18,2.13,76.45,-0.44,76.00
Kuwait,82.50,80.27,86.43,6.16,4.92,70.48,69.73,71.79,2.06,3.23,86.68,-1.25,85.43,25.47,23.77,28.37,4.60,3.34,18.66,17.64,20.36,2.72,2.04,75.10,-1.84,73.26
Cyprus,82.18,80.20,84.16,3.96,3.35,71.13,70.69,71.55,0.86,2.66,86.86,-0.30,86.55,24.36,22.91,25.77,2.86,2.53,18.56,17.79,19.31,1.52,1.93,76.18,0.01,76.19
United Arab Emirates,81.41,80.98,82.29,1.31,3.29,69.69,70.24,68.69,-1.55,2.24,86.34,-0.74,85.60,23.55,23.52,23.68,0.16,2.17,17.29,17.59,16.95,-0.64,1.39,74.37,-0.95,73.42
Jordan,79.79,79.73,80.34,0.61,7.07,68.36,69.59,67.32,-2.27,5.11,86.98,-1.30,85.67,23.48,24.08,23.36,-0.72,4.75,17.33,18.03,16.95,-1.08,3.17,75.60,-1.79,73.81
Lebanon,79.29,76.92,81.47,4.55,3.57,67.41,66.73,68.03,1.30,2.41,85.84,-0.83,85.02,23.18,21.46,24.66,3.20,2.46,16.93,15.83,17.88,2.05,1.40,74.95,-1.91,73.04


In [24]:
table_code_Asia = create_table_countries(df_Asia, lang='en')
output_table_code(table_code_Asia, 'Table code WHO -Asia -en.txt', destination=DESTINATION_OUTPUT)

Data has written to file


In [25]:
table_code_Asia = create_table_countries(df_Asia, lang='ru')
output_table_code(table_code_Asia, 'Table code WHO -Asia -ru.txt', destination=DESTINATION_OUTPUT)

Data has written to file


<br />
<br />

In [27]:
ls_Oceania = ['Australia', 'Fiji', 'Kiribati', 'Micronesia', 'New Zealand', 'Papua New Guinea',
              'Samoa', 'Solomon Islands', 'Tonga', 'Vanuatu',
              'World', 'Western Pacific']

# WHO list does not contain 'Marshall Islands', 'Nauru', 'Tuvalu', 'French Polynesia', 'New Caledonia', 'Guam'

print("Number of records for Oceania:", len(ls_Oceania))

df_Oceania = df.loc[ls_Oceania]   \
              .sort_values(by=['LE_birth_o', 'HALE_birth_o', 'LE_60_o', 'HALE_60_o'], ascending=False)
df_Oceania

Number of records for Oceania: 12


,LE_birth_o,LE_birth_m,LE_birth_f,LE_birth_fΔm,LE_birth_Δ_2000,HALE_birth_o,HALE_birth_m,HALE_birth_f,HALE_birth_fΔm,HALE_birth_Δ_2000,ratio_birth_2000,ratio_birth_Δ,ratio_birth,LE_60_o,LE_60_m,LE_60_f,LE_60_fΔm,LE_60_Δ_2000,HALE_60_o,HALE_60_m,HALE_60_f,HALE_60_fΔm,HALE_60_Δ_2000,ratio_60_2000,ratio_60_Δ,ratio_60
,,,,,,,,,,,,,,,,,,,,,,,,,,
Australia,82.64,80.72,84.57,3.85,2.94,70.26,69.68,70.84,1.16,2.15,85.46,-0.44,85.02,25.35,24.05,26.60,2.55,2.24,18.81,18.08,19.53,1.45,1.45,75.12,-0.92,74.20
New Zealand,81.81,80.16,83.43,3.27,3.23,69.74,69.50,69.98,0.48,2.47,85.61,-0.36,85.25,24.72,23.60,25.77,2.17,2.34,18.52,17.95,19.07,1.12,1.62,75.51,-0.59,74.92
Western Pacific,77.49,74.51,80.70,6.19,5.53,68.35,66.70,70.13,3.43,4.53,88.69,-0.48,88.20,21.75,19.68,23.84,4.16,2.73,16.64,15.38,17.90,2.52,1.90,77.50,-0.99,76.51
World,73.12,70.61,75.70,5.09,6.35,63.45,62.33,64.59,2.26,5.33,87.05,-0.27,86.78,21.03,19.41,22.54,3.13,2.16,15.80,14.87,16.67,1.80,1.52,75.68,-0.54,75.13
Tonga,72.94,70.38,75.60,5.22,1.99,64.05,62.97,65.21,2.24,1.34,88.39,-0.57,87.81,19.04,17.41,20.65,3.24,1.15,14.44,13.55,15.32,1.77,0.70,76.80,-0.96,75.84
Samoa,69.99,68.77,71.31,2.54,0.39,61.42,61.11,61.77,0.66,0.16,88.02,-0.26,87.76,17.64,17.22,18.06,0.84,0.56,13.30,13.20,13.41,0.21,0.32,76.00,-0.60,75.40
Fiji,67.87,65.78,70.09,4.31,2.04,59.67,58.64,60.77,2.13,1.54,88.30,-0.39,87.92,16.01,14.71,17.27,2.56,1.47,11.96,11.13,12.76,1.63,0.94,75.79,-1.09,74.70
Vanuatu,67.13,64.12,70.64,6.52,2.20,59.38,57.62,61.43,3.81,1.89,88.54,-0.09,88.46,16.18,15.05,17.63,2.58,0.93,12.47,11.80,13.31,1.51,0.67,77.38,-0.31,77.07
Papua New Guinea,66.67,65.57,67.92,2.35,3.31,58.47,58.09,58.92,0.83,2.90,87.71,-0.00,87.70,16.77,16.34,17.28,0.94,0.75,12.68,12.48,12.92,0.44,0.51,75.97,-0.36,75.61


In [28]:
table_code_Oceania = create_table_countries(df_Oceania, lang='en')
output_table_code(table_code_Oceania, 'Table code WHO -Oceania -en.txt', destination=DESTINATION_OUTPUT)

Data has written to file


In [29]:
table_code_Oceania = create_table_countries(df_Oceania, lang='ru')
output_table_code(table_code_Oceania, 'Table code WHO -Oceania -ru.txt', destination=DESTINATION_OUTPUT)

Data has written to file


<br />
<br />

In [31]:
ls_N_America = ['Antigua and Barbuda', 'Bahamas', 'Barbados', 'Belize', 'Canada', 'Costa Rica',
                'Cuba', 'Dominican Republic', 'El Salvador', 'Grenada', 'Guatemala', 'Haiti',
                'Honduras', 'Jamaica', 'Mexico', 'Nicaragua', 'Panama', 'Puerto Rico',
                'Saint Lucia', 'St. Vincent and the Grenadines', 'Trinidad and Tobago', 'USA',
                'World', 'Americas']

# small countries: Dominica, Saint Kitts and Nevis

print("Number of records for North America:", len(ls_N_America))

df_N_America = df.loc[ls_N_America]   \
                 .sort_values(by=['LE_birth_o', 'HALE_birth_o', 'LE_60_o', 'HALE_60_o'], ascending=False)
df_N_America

Number of records for North America: 24


,LE_birth_o,LE_birth_m,LE_birth_f,LE_birth_fΔm,LE_birth_Δ_2000,HALE_birth_o,HALE_birth_m,HALE_birth_f,HALE_birth_fΔm,HALE_birth_Δ_2000,ratio_birth_2000,ratio_birth_Δ,ratio_birth,LE_60_o,LE_60_m,LE_60_f,LE_60_fΔm,LE_60_Δ_2000,HALE_60_o,HALE_60_m,HALE_60_f,HALE_60_fΔm,HALE_60_Δ_2000,ratio_60_2000,ratio_60_Δ,ratio_60
,,,,,,,,,,,,,,,,,,,,,,,,,,
Canada,82.02,80.12,83.90,3.78,2.93,70.30,69.66,70.91,1.25,1.76,86.66,-0.95,85.71,24.99,23.66,26.23,2.57,2.52,18.66,17.89,19.38,1.49,1.58,76.01,-1.34,74.67
Puerto Rico,80.50,76.86,84.04,7.18,4.25,69.44,67.20,71.62,4.42,3.24,86.82,-0.56,86.26,25.32,23.46,26.98,3.52,3.21,19.07,17.74,20.26,2.52,2.17,76.44,-1.12,75.32
Costa Rica,80.30,77.93,82.71,4.78,2.13,69.21,68.09,70.35,2.26,1.38,86.77,-0.58,86.19,24.52,23.11,25.85,2.74,1.97,18.37,17.49,19.20,1.71,1.25,75.92,-1.00,74.92
Panama,78.97,76.20,81.86,5.66,2.82,68.31,66.89,69.79,2.90,2.06,87.00,-0.50,86.50,23.98,22.14,25.80,3.66,1.85,18.08,16.88,19.26,2.38,1.18,76.37,-0.97,75.40
Nicaragua,78.87,76.30,81.27,4.97,2.69,67.68,66.26,69.00,2.74,2.39,85.70,0.11,85.81,24.50,23.23,25.51,2.28,0.46,18.19,17.38,18.83,1.45,0.22,74.75,-0.51,74.24
USA,78.74,76.53,80.98,4.45,2.08,66.02,65.13,66.93,1.80,0.70,85.21,-1.36,83.85,23.24,21.99,24.41,2.42,1.90,16.59,15.82,17.31,1.49,0.92,73.43,-2.04,71.39
Cuba,77.66,75.30,80.12,4.82,0.91,67.80,66.74,68.88,2.14,0.78,87.32,-0.02,87.30,21.57,19.96,23.15,3.19,-0.04,16.55,15.53,17.55,2.02,-0.17,77.37,-0.64,76.73
Americas,77.07,74.42,79.76,5.34,2.98,65.76,64.51,67.01,2.50,2.19,85.80,-0.48,85.33,22.64,21.23,23.92,2.69,1.62,16.61,15.74,17.40,1.66,0.96,74.45,-1.09,73.37
Saint Lucia,76.12,72.53,79.93,7.40,2.90,65.71,63.71,67.84,4.13,2.18,86.77,-0.44,86.32,22.16,19.63,24.67,5.04,2.18,16.73,14.96,18.49,3.53,1.51,76.18,-0.68,75.50


In [32]:
table_code_N_America = create_table_countries(df_N_America, lang='en')
output_table_code(table_code_N_America, 'Table code WHO -N_America -en.txt', destination=DESTINATION_OUTPUT)

Data has written to file


In [33]:
table_code_N_America = create_table_countries(df_N_America, lang='ru')
output_table_code(table_code_N_America, 'Table code WHO -N_America -ru.txt', destination=DESTINATION_OUTPUT)

Data has written to file


<br />
<br />

In [35]:
ls_S_America = ['Argentina', 'Bolivia', 'Brazil', 'Chile', 'Colombia', 'Ecuador',
                'Guyana', 'Paraguay', 'Peru', 'Suriname', 'Uruguay', 'Venezuela',
                'World', 'Americas']

# main countries: Brazil, Colombia, Argentina, Peru, Chile  # Venezuela

print("Number of records for South America:", len(ls_S_America))

df_S_America = df.loc[ls_S_America]   \
                 .sort_values(by=['LE_birth_o', 'HALE_birth_o', 'LE_60_o', 'HALE_60_o'], ascending=False)
df_S_America

Number of records for South America: 14


,LE_birth_o,LE_birth_m,LE_birth_f,LE_birth_fΔm,LE_birth_Δ_2000,HALE_birth_o,HALE_birth_m,HALE_birth_f,HALE_birth_fΔm,HALE_birth_Δ_2000,ratio_birth_2000,ratio_birth_Δ,ratio_birth,LE_60_o,LE_60_m,LE_60_f,LE_60_fΔm,LE_60_Δ_2000,HALE_60_o,HALE_60_m,HALE_60_f,HALE_60_fΔm,HALE_60_Δ_2000,ratio_60_2000,ratio_60_Δ,ratio_60
,,,,,,,,,,,,,,,,,,,,,,,,,,
Chile,81.03,78.66,83.34,4.68,4.23,69.37,68.65,70.05,1.40,3.11,86.28,-0.67,85.61,24.57,23.01,25.97,2.96,3.25,18.34,17.48,19.10,1.62,2.11,76.13,-1.48,74.64
Peru,78.29,76.69,79.88,3.19,3.17,68.41,67.97,68.86,0.89,2.99,87.09,0.29,87.38,23.37,22.59,24.10,1.51,0.56,18.02,17.65,18.37,0.72,0.47,76.94,0.17,77.11
Colombia,77.95,75.33,80.55,5.22,5.41,67.90,66.44,69.34,2.90,4.58,87.29,-0.18,87.11,22.69,21.20,24.03,2.83,2.40,17.41,16.42,18.31,1.89,1.80,76.93,-0.20,76.73
Ecuador,77.75,75.32,80.23,4.91,5.16,67.62,66.42,68.83,2.41,4.12,87.48,-0.51,86.97,22.83,21.46,24.12,2.66,2.27,17.36,16.50,18.17,1.67,1.51,77.09,-1.05,76.04
Americas,77.07,74.42,79.76,5.34,2.98,65.76,64.51,67.01,2.50,2.19,85.80,-0.48,85.33,22.64,21.23,23.92,2.69,1.62,16.61,15.74,17.40,1.66,0.96,74.45,-1.09,73.37
Argentina,77.02,74.00,79.91,5.91,2.73,66.90,65.43,68.27,2.84,2.17,87.13,-0.27,86.86,21.59,19.24,23.64,4.40,1.28,16.50,14.99,17.81,2.82,0.86,77.01,-0.58,76.42
Uruguay,77.01,73.36,80.50,7.14,2.13,66.81,64.91,68.60,3.69,1.47,87.26,-0.50,86.75,21.71,18.95,24.04,5.09,1.02,16.51,14.71,18.02,3.31,0.59,76.95,-0.90,76.05
Brazil,75.48,72.22,78.73,6.51,3.98,64.53,62.96,66.07,3.11,3.61,85.20,0.29,85.49,21.32,19.58,22.84,3.26,1.54,15.81,14.74,16.74,2.00,1.13,74.22,-0.06,74.16
Paraguay,75.08,72.18,78.16,5.98,0.40,64.87,63.47,66.38,2.91,0.46,86.25,0.15,86.40,21.01,19.22,22.82,3.60,-1.09,15.88,14.73,17.04,2.31,-0.85,75.70,-0.12,75.58


In [36]:
table_code_S_America = create_table_countries(df_S_America, lang='en')
output_table_code(table_code_S_America, 'Table code WHO -S_America -en.txt', destination=DESTINATION_OUTPUT)

Data has written to file


In [37]:
table_code_S_America = create_table_countries(df_S_America, lang='ru')
output_table_code(table_code_S_America, 'Table code WHO -S_America -ru.txt', destination=DESTINATION_OUTPUT)

Data has written to file


<br />
<br />

In [39]:
ls_Americas = ls_N_America + ls_S_America
ls_Americas.remove('World')
ls_Americas.remove('Americas')

print("Number of records for the Americas:", len(ls_Americas))

df_Americas = df.loc[ls_Americas]   \
                 .sort_values(by=['LE_birth_o', 'HALE_birth_o', 'LE_60_o', 'HALE_60_o'], ascending=False)
df_Americas

Number of records for the Americas: 36


,LE_birth_o,LE_birth_m,LE_birth_f,LE_birth_fΔm,LE_birth_Δ_2000,HALE_birth_o,HALE_birth_m,HALE_birth_f,HALE_birth_fΔm,HALE_birth_Δ_2000,ratio_birth_2000,ratio_birth_Δ,ratio_birth,LE_60_o,LE_60_m,LE_60_f,LE_60_fΔm,LE_60_Δ_2000,HALE_60_o,HALE_60_m,HALE_60_f,HALE_60_fΔm,HALE_60_Δ_2000,ratio_60_2000,ratio_60_Δ,ratio_60
,,,,,,,,,,,,,,,,,,,,,,,,,,
Canada,82.02,80.12,83.90,3.78,2.93,70.30,69.66,70.91,1.25,1.76,86.66,-0.95,85.71,24.99,23.66,26.23,2.57,2.52,18.66,17.89,19.38,1.49,1.58,76.01,-1.34,74.67
Chile,81.03,78.66,83.34,4.68,4.23,69.37,68.65,70.05,1.40,3.11,86.28,-0.67,85.61,24.57,23.01,25.97,2.96,3.25,18.34,17.48,19.10,1.62,2.11,76.13,-1.48,74.64
Puerto Rico,80.50,76.86,84.04,7.18,4.25,69.44,67.20,71.62,4.42,3.24,86.82,-0.56,86.26,25.32,23.46,26.98,3.52,3.21,19.07,17.74,20.26,2.52,2.17,76.44,-1.12,75.32
Costa Rica,80.30,77.93,82.71,4.78,2.13,69.21,68.09,70.35,2.26,1.38,86.77,-0.58,86.19,24.52,23.11,25.85,2.74,1.97,18.37,17.49,19.20,1.71,1.25,75.92,-1.00,74.92
Panama,78.97,76.20,81.86,5.66,2.82,68.31,66.89,69.79,2.90,2.06,87.00,-0.50,86.50,23.98,22.14,25.80,3.66,1.85,18.08,16.88,19.26,2.38,1.18,76.37,-0.97,75.40
Nicaragua,78.87,76.30,81.27,4.97,2.69,67.68,66.26,69.00,2.74,2.39,85.70,0.11,85.81,24.50,23.23,25.51,2.28,0.46,18.19,17.38,18.83,1.45,0.22,74.75,-0.51,74.24
USA,78.74,76.53,80.98,4.45,2.08,66.02,65.13,66.93,1.80,0.70,85.21,-1.36,83.85,23.24,21.99,24.41,2.42,1.90,16.59,15.82,17.31,1.49,0.92,73.43,-2.04,71.39
Peru,78.29,76.69,79.88,3.19,3.17,68.41,67.97,68.86,0.89,2.99,87.09,0.29,87.38,23.37,22.59,24.10,1.51,0.56,18.02,17.65,18.37,0.72,0.47,76.94,0.17,77.11
Colombia,77.95,75.33,80.55,5.22,5.41,67.90,66.44,69.34,2.90,4.58,87.29,-0.18,87.11,22.69,21.20,24.03,2.83,2.40,17.41,16.42,18.31,1.89,1.80,76.93,-0.20,76.73


In [40]:
table_code_Americas = create_table_countries(df_Americas, lang='en')
output_table_code(table_code_Americas, 'Table code WHO -Americas -en.txt', destination=DESTINATION_OUTPUT)

Data has written to file


In [41]:
table_code_Americas = create_table_countries(df_Americas, lang='ru')
output_table_code(table_code_Americas, 'Table code WHO -Americas -ru.txt', destination=DESTINATION_OUTPUT)

Data has written to file


<br />
<br />

In [43]:
ls_Africa = ['Algeria', 'Angola', 'Benin', 'Botswana', 'Burkina Faso', 'Burundi', 'Cameroon', 'Cape Verde',
             'Central African Republic', 'Chad', 'Comoros', 'Congo DR', 'Congo Republic', "Cote d'Ivoire",
             'Djibouti', 'Egypt', 'Equatorial Guinea', 'Eritrea', 'Eswatini', 'Ethiopia', 'Gabon', 'Gambia',
             'Ghana', 'Guinea', 'Guinea-Bissau', 'Kenya', 'Lesotho', 'Liberia', 'Libya', 'Madagascar', 'Malawi',
             'Mali', 'Mauritania', 'Mauritius', 'Morocco', 'Mozambique', 'Namibia', 'Niger', 'Nigeria', 'Rwanda',
             'Sao Tome and Principe', 'Senegal', 'Seychelles', 'Sierra Leone', 'Somalia', 'South Africa',
             'South Sudan', 'Sudan', 'Tanzania', 'Togo', 'Tunisia', 'Uganda', 'Zambia', 'Zimbabwe',
             'World', 'Africa']

# main countries: Nigeria, Ethiopia, Egypt, Democratic Republic of the Congo, South Africa, Algeria, Tanzania, Kenya

print("Number of records for Africa:", len(ls_Africa))

df_Africa = df.loc[ls_Africa]   \
              .sort_values(by=['LE_birth_o', 'HALE_birth_o', 'LE_60_o', 'HALE_60_o'], ascending=False)
df_Africa

Number of records for Africa: 56


,LE_birth_o,LE_birth_m,LE_birth_f,LE_birth_fΔm,LE_birth_Δ_2000,HALE_birth_o,HALE_birth_m,HALE_birth_f,HALE_birth_fΔm,HALE_birth_Δ_2000,ratio_birth_2000,ratio_birth_Δ,ratio_birth,LE_60_o,LE_60_m,LE_60_f,LE_60_fΔm,LE_60_Δ_2000,HALE_60_o,HALE_60_m,HALE_60_f,HALE_60_fΔm,HALE_60_Δ_2000,ratio_60_2000,ratio_60_Δ,ratio_60
,,,,,,,,,,,,,,,,,,,,,,,,,,
Tunisia,77.51,75.09,79.99,4.90,3.21,66.68,66.03,67.38,1.35,2.33,86.61,-0.58,86.03,21.74,20.14,23.33,3.19,1.47,16.19,15.26,17.12,1.86,0.89,75.48,-1.01,74.47
Algeria,76.55,76.24,77.08,0.84,4.08,66.14,67.04,65.34,-1.70,3.20,86.85,-0.45,86.40,21.43,21.62,21.47,-0.15,1.39,16.09,16.45,15.89,-0.56,0.82,76.20,-1.12,75.08
Cape Verde,74.06,69.71,78.23,8.52,2.36,64.76,62.30,67.06,4.76,2.32,87.09,0.36,87.44,19.59,16.54,21.95,5.41,-1.09,14.91,12.93,16.44,3.51,-0.90,76.45,-0.34,76.11
Mauritius,74.06,70.83,77.43,6.60,2.92,64.25,62.46,66.12,3.66,2.06,87.42,-0.67,86.75,20.40,18.56,22.09,3.53,2.59,15.08,13.99,16.08,2.09,1.65,75.41,-1.49,73.92
Morocco,73.75,72.85,74.64,1.79,3.56,63.60,64.13,63.04,-1.09,2.91,86.47,-0.23,86.24,18.97,18.32,19.59,1.27,-0.09,14.15,13.92,14.36,0.44,-0.32,75.92,-1.33,74.59
Libya,73.54,71.47,75.70,4.23,-0.50,63.36,62.73,63.99,1.26,-0.87,86.75,-0.59,86.16,19.90,19.10,20.63,1.53,-1.02,14.76,14.36,15.13,0.77,-1.08,75.72,-1.55,74.17
Seychelles,73.12,70.08,76.65,6.57,1.01,64.31,62.52,66.37,3.85,0.62,88.32,-0.37,87.95,19.38,17.51,21.23,3.72,0.19,14.54,13.34,15.75,2.41,-0.12,76.39,-1.37,75.03
World,73.12,70.61,75.70,5.09,6.35,63.45,62.33,64.59,2.26,5.33,87.05,-0.27,86.78,21.03,19.41,22.54,3.13,2.16,15.80,14.87,16.67,1.80,1.52,75.68,-0.54,75.13
Sao Tome and Principe,71.66,70.16,73.18,3.02,8.53,62.95,62.80,63.09,0.29,7.43,87.95,-0.10,87.85,17.77,16.82,18.69,1.87,0.83,13.62,13.21,14.02,0.81,0.58,76.98,-0.33,76.65


In [44]:
table_code_Africa = create_table_countries(df_Africa, lang='en')
output_table_code(table_code_Africa, 'Table code WHO -Africa -en.txt', destination=DESTINATION_OUTPUT)

Data has written to file


In [45]:
table_code_Africa = create_table_countries(df_Africa, lang='ru')
output_table_code(table_code_Africa, 'Table code WHO -Africa -ru.txt', destination=DESTINATION_OUTPUT)

Data has written to file


<br />
<br />
<br />

[Demographics of the European Union](https://en.wikipedia.org/wiki/Demographics_of_the_European_Union)<br>
[Население стран Европейского союза](https://ru.wikipedia.org/wiki/Население_стран_Европейского_союза)

In [47]:
ls_EU =  ['Austria', 'Belgium', 'Bulgaria', 'Croatia', 'Cyprus', 'Czechia', 'Denmark', 'Estonia',
          'Finland', 'France', 'Germany', 'Greece', 'Hungary', 'Ireland', 'Italy', 'Latvia',
          'Lithuania', 'Luxembourg', 'Malta', 'Netherlands', 'Poland', 'Portugal', 'Romania',
          'Slovakia', 'Slovenia', 'Spain', 'Sweden', 'United Kingdom',
          'World']  #  - UK left EU 2020 January 31

print("Number of records for the European Union:", len(ls_EU))

df_EU = df.loc[ls_EU]   \
          .sort_values(by=['LE_birth_o', 'HALE_birth_o', 'LE_60_o', 'HALE_60_o'], ascending=False)
df_EU

Number of records for the European Union: 29


,LE_birth_o,LE_birth_m,LE_birth_f,LE_birth_fΔm,LE_birth_Δ_2000,HALE_birth_o,HALE_birth_m,HALE_birth_f,HALE_birth_fΔm,HALE_birth_Δ_2000,ratio_birth_2000,ratio_birth_Δ,ratio_birth,LE_60_o,LE_60_m,LE_60_f,LE_60_fΔm,LE_60_Δ_2000,HALE_60_o,HALE_60_m,HALE_60_f,HALE_60_fΔm,HALE_60_Δ_2000,ratio_60_2000,ratio_60_Δ,ratio_60
,,,,,,,,,,,,,,,,,,,,,,,,,,
Spain,83.14,80.54,85.66,5.12,4.05,71.69,70.90,72.42,1.52,2.80,87.10,-0.88,86.23,25.39,23.28,27.33,4.05,2.69,19.18,17.94,20.33,2.39,1.78,76.65,-1.11,75.54
Italy,82.99,80.94,84.91,3.97,3.62,71.43,70.85,71.92,1.07,3.04,86.17,-0.10,86.07,25.14,23.53,26.56,3.03,2.60,18.95,18.03,19.78,1.75,1.94,75.47,-0.09,75.38
Luxembourg,82.80,80.83,84.74,3.91,4.38,71.46,70.98,71.89,0.91,3.61,86.52,-0.22,86.30,24.73,23.13,26.22,3.09,2.85,18.83,17.97,19.64,1.67,2.10,76.46,-0.32,76.14
Sweden,82.71,81.17,84.24,3.07,3.14,71.41,71.33,71.46,0.13,2.15,87.04,-0.71,86.34,24.78,23.57,25.91,2.34,2.36,18.80,18.26,19.32,1.06,1.57,76.85,-0.98,75.87
France,82.53,79.77,85.15,5.38,3.62,70.72,69.78,71.60,1.82,2.74,86.15,-0.46,85.69,25.69,23.63,27.51,3.88,2.68,19.26,18.07,20.33,2.26,1.89,75.49,-0.52,74.97
Netherlands,82.34,80.85,83.76,2.91,4.33,71.08,71.05,71.07,0.02,2.98,87.30,-0.97,86.32,24.65,23.43,25.76,2.33,3.33,18.64,18.07,19.16,1.09,2.16,77.30,-1.68,75.62
Cyprus,82.18,80.20,84.16,3.96,3.35,71.13,70.69,71.55,0.86,2.66,86.86,-0.30,86.55,24.36,22.91,25.77,2.86,2.53,18.56,17.79,19.31,1.52,1.93,76.18,0.01,76.19
Malta,82.15,80.31,83.98,3.67,4.33,71.16,70.80,71.43,0.63,3.14,87.41,-0.78,86.62,24.59,23.06,26.03,2.97,3.77,18.91,18.07,19.70,1.63,2.66,78.05,-1.15,76.90
Ireland,81.94,80.27,83.60,3.33,5.52,70.48,70.25,70.71,0.46,4.16,86.78,-0.77,86.01,24.26,23.00,25.48,2.48,4.07,18.53,17.90,19.14,1.24,2.97,77.07,-0.69,76.38


In [48]:
table_code_EU = create_table_countries(df_EU, lang='en')
output_table_code(table_code_EU, 'Table code WHO -EU -en.txt', destination=DESTINATION_OUTPUT)

Data has written to file


In [49]:
table_code_EU = create_table_countries(df_EU, lang='ru')
output_table_code(table_code_EU, 'Table code WHO -EU -ru.txt', destination=DESTINATION_OUTPUT)

Data has written to file


<br />
<br />
<br />

[CIS](https://en.wikipedia.org/wiki/Commonwealth_of_Independent_States#Life_expectancy) /
[СНГ](https://ru.wikipedia.org/wiki/Содружество_Независимых_Государств#Социальное_развитие_стран_СНГ)

In [51]:
ls_CIS = ['Armenia', 'Azerbaijan', 'Belarus', 'Kazakhstan', 'Kyrgyzstan',
          'Moldova', 'Russia', 'Tajikistan', 'Turkmenistan', 'Uzbekistan',
          'World']    # Georgia and Ukraine are former CIS members

print("Number of records for the Commonwealth of Independent States:", len(ls_CIS))

df_CIS = df.loc[ls_CIS]   \
           .sort_values(by=['LE_birth_o', 'HALE_birth_o', 'LE_60_o', 'HALE_60_o'], ascending=False)
df_CIS

Number of records for the Commonwealth of Independent States: 11


,LE_birth_o,LE_birth_m,LE_birth_f,LE_birth_fΔm,LE_birth_Δ_2000,HALE_birth_o,HALE_birth_m,HALE_birth_f,HALE_birth_fΔm,HALE_birth_Δ_2000,ratio_birth_2000,ratio_birth_Δ,ratio_birth,LE_60_o,LE_60_m,LE_60_f,LE_60_fΔm,LE_60_Δ_2000,HALE_60_o,HALE_60_m,HALE_60_f,HALE_60_fΔm,HALE_60_Δ_2000,ratio_60_2000,ratio_60_Δ,ratio_60
,,,,,,,,,,,,,,,,,,,,,,,,,,
Azerbaijan,75.80,73.38,78.09,4.71,10.57,66.62,65.57,67.63,2.06,8.81,88.62,-0.74,87.89,20.86,19.55,21.95,2.40,4.86,16.15,15.49,16.72,1.23,3.54,78.81,-1.39,77.42
Armenia,75.67,70.82,79.79,8.97,4.05,66.48,63.56,68.96,5.40,3.51,87.92,-0.07,87.86,20.10,16.81,22.61,5.80,2.12,15.45,13.30,17.09,3.79,1.56,77.25,-0.39,76.87
Belarus,74.82,69.73,79.66,9.93,6.03,65.44,62.14,68.56,6.42,5.23,87.53,-0.06,87.46,19.67,16.07,22.49,6.42,2.55,14.88,12.30,16.90,4.60,1.93,75.64,0.01,75.65
Tajikistan,73.81,72.28,75.37,3.09,8.16,64.91,64.42,65.42,1.00,7.12,88.03,-0.09,87.94,20.10,19.57,20.62,1.05,2.73,15.67,15.54,15.80,0.26,2.07,78.30,-0.34,77.96
Kyrgyzstan,73.59,69.92,77.14,7.22,7.66,64.84,62.68,66.93,4.25,6.73,88.14,-0.03,88.11,19.46,17.19,21.40,4.21,2.91,15.13,13.69,16.37,2.68,2.29,77.58,0.17,77.75
Moldova,73.32,69.32,77.17,7.85,7.66,64.43,62.01,66.75,4.74,6.41,88.36,-0.49,87.88,19.02,16.63,20.92,4.29,4.09,14.49,12.87,15.79,2.92,3.01,76.89,-0.71,76.18
Russia,73.22,68.17,78.00,9.83,8.06,63.72,60.41,66.84,6.43,7.01,87.03,-0.01,87.03,19.93,16.80,22.19,5.39,3.54,14.92,12.75,16.49,3.74,2.64,74.92,-0.06,74.86
Kazakhstan,73.20,68.84,77.20,8.36,8.85,63.97,61.26,66.45,5.19,7.60,87.60,-0.21,87.39,19.15,16.43,21.16,4.73,3.25,14.49,12.75,15.77,3.02,2.39,76.10,-0.43,75.67
World,73.12,70.61,75.70,5.09,6.35,63.45,62.33,64.59,2.26,5.33,87.05,-0.27,86.78,21.03,19.41,22.54,3.13,2.16,15.80,14.87,16.67,1.80,1.52,75.68,-0.54,75.13


In [52]:
# table_code_CIS = create_table_countries(df_CIS, lang='en')
# output_table_code(table_code_CIS, 'Table code WHO -CIS -en.txt', destination=DESTINATION_OUTPUT)

In [53]:
table_code_CIS = create_table_countries(df_CIS, lang='ru')
output_table_code(table_code_CIS, 'Table code WHO -CIS -ru.txt', destination=DESTINATION_OUTPUT)

Data has written to file


<br />
<br />

In [55]:
# create code for placing info about countries in Wikipedia (for countries)
def create_table_regions(df, lang='en', file_header=None):

    def if_value(x, prec=2):
        return '—' if math.isnan(x) else \
               f"{x:0.{prec}f}"  if x>=0 else \
               f"−{-x:0.{prec}f}"

    if lang=='ru':
        file_header='who_stats_header_regions_ru.txt'
        prettify_name = {
            "World": "\'\'\'Мир\'\'\'",
            "Europe": "[[Европа]]<ref>{{cite web|title=ВОЗ: Европа |publisher=Всемирная организация здравоохранения |url=https://www.who.int/europe/ru/ |access-date=2025-10-18}}</ref>",
            "Eastern Mediterranean": "Восточное Средиземноморье<ref>{{cite web|title=WHO: Eastern Mediterranean: Countries |lang=en |publisher=Всемирная организация здравоохранения |url=http://www.emro.who.int/countries.html |access-date=2025-10-18}}</ref>",
            "South-East Asia": "[[Юго-Восточная Азия]]<ref>{{cite web|title=WHO: South-East Asia: Where we work |lang=en |publisher=Всемирная организация здравоохранения |url=https://www.who.int/southeastasia/about/where-we-work |access-date=2025-10-18}}</ref>",
            "Western Pacific": "Западно-тихоокеанский регион<ref>{{cite web|title=WHO: Western Pacific: Where we work |lang=en |publisher=Всемирная организация здравоохранения |url=https://www.who.int/westernpacific/about/where-we-work |access-date=2025-10-18}}</ref>",
            "Americas": "[[Америка]]<ref>{{cite web|title=WHO: PAHO: Countries and Centers |lang=en |publisher=Всемирная организация здравоохранения |url=https://www.paho.org/en/countries-and-centers |access-date=2025-10-18}}</ref>",
            "Africa": "[[Африка]]<ref>{{cite web|title=WHO: Africa: Countries |lang=en |publisher=Всемирная организация здравоохранения |url=https://www.afro.who.int/countries |access-date=2025-10-18}}</ref>",
            "High-income": "Высокий доход",
            "Upper-middle-income": "Средне-высокий доход",
            "Lower-middle-income": "Средне-низкий доход",
            "Low-income": "Низкий доход"
        }
    else:
        file_header='who_stats_header_regions_en.txt'
        prettify_name = {
            "World": "\'\'\'World\'\'\'",
            "Europe": "[[Europe]]<ref>{{cite web|title=WHO: Europe |publisher=World Health Organization |url=https://www.who.int/europe |access-date=18 November 2025}}</ref>",
            "Eastern Mediterranean": "[[Eastern Mediterranean]]<ref>{{cite web|title=WHO: Eastern Mediterranean: Countries |publisher=World Health Organization |url=http://www.emro.who.int/countries.html |access-date=18 November 2025}}</ref>",
            "South-East Asia": "[[South-East Asia]]<ref>{{cite web|title=WHO: South-East Asia: Where we work |language=en |publisher=World Health Organization |url=https://www.who.int/southeastasia/about/where-we-work |access-date=18 November 2025}}</ref>",
            "Western Pacific": "Western Pacific<ref>{{cite web|title=WHO: Western Pacific: Where we work |publisher=World Health Organization |url=https://www.who.int/westernpacific/about/where-we-work |access-date=18 November 2025}}</ref>",
            "Americas": "[[Americas]]<ref>{{cite web|title=WHO: PAHO: Countries and Centers |publisher=World Health Organization |url=https://www.paho.org/en/countries-and-centers |access-date=18 November 2025}}</ref>",
            "Africa": "[[Africa]]<ref>{{cite web|title=WHO: Africa: Countries |language=en |publisher=World Health Organization |url=https://www.afro.who.int/countries |access-date=18 November 2025}}</ref>",
        } 
            

    with open('design/' + file_header, mode='r', encoding="utf-8") as fh:
        st_header = fh.read()
        
    st_header = st_header.strip()
    st = ''
        
    for i in range(len(df)):
        ser = df.iloc[i]
        
        comp_LE_birth = ser.LE_birth_Δ_2000
        comp_HALE_birth = ser.HALE_birth_Δ_2000
        comp_LE_60 = ser.LE_60_Δ_2000
        comp_HALE_60 = ser.HALE_60_Δ_2000
        
        if ser.name in ['World']:
            st += '\n' + '|-class=static-row-header\n' + \
                  f'|style="text-align:center"| {prettify_name.get(ser.name, ser.name)} ' + \
                  f'||style="background:#e0ffd8;"| \'\'\'{ser.LE_birth_o:0.2f}\'\'\' ' + \
                  f'||style="background:#eaf3ff;"| \'\'\'{ser.LE_birth_m:0.2f}\'\'\' ' + \
                  f'||style="background:#fee7f6;"| \'\'\'{ser.LE_birth_f:0.2f}\'\'\' ' + \
                  f'|| \'\'\'{if_value(ser.LE_birth_fΔm, 2)}\'\'\' ' + \
                  f'||style="background:#fff8dc;"| \'\'\'{if_value(comp_LE_birth, 2)}\'\'\' ' + \
                  f'||style="background:#e0ffd8; border-left-width:2px;"| \'\'\'{ser.HALE_birth_o:0.2f}\'\'\' ' + \
                  f'||style="background:#eaf3ff;"| \'\'\'{ser.HALE_birth_m:0.2f}\'\'\' ' + \
                  f'||style="background:#fee7f6;"| \'\'\'{ser.HALE_birth_f:0.2f}\'\'\' ' + \
                  f'|| \'\'\'{if_value(ser.HALE_birth_fΔm, 2)}\'\'\' ' + \
                  f'||style="background:#fff8dc;"| \'\'\'{if_value(comp_HALE_birth, 2)}\'\'\' ' + \
                  f'||style="background:#e0ffd8; border-left-width:3px;"| \'\'\'{ser.LE_60_o:0.2f}\'\'\' ' + \
                  f'||style="background:#eaf3ff;"| \'\'\'{ser.LE_60_m:0.2f}\'\'\' ' + \
                  f'||style="background:#fee7f6;"| \'\'\'{ser.LE_60_f:0.2f}\'\'\' ' + \
                  f'|| \'\'\'{if_value(ser.LE_60_fΔm, 2)}\'\'\' ' + \
                  f'||style="background:#fff8dc;"| \'\'\'{if_value(comp_LE_60, 2)}\'\'\' ' + \
                  f'||style="background:#e0ffd8; border-left-width:2px;"| \'\'\'{ser.HALE_60_o:0.2f}\'\'\' ' + \
                  f'||style="background:#eaf3ff;"| \'\'\'{ser.HALE_60_m:0.2f}\'\'\' ' + \
                  f'||style="background:#fee7f6;"| \'\'\'{ser.HALE_60_f:0.2f}\'\'\' ' + \
                  f'|| \'\'\'{if_value(ser.HALE_60_fΔm, 2)}\'\'\' ' + \
                  f'||style="background:#fff8dc;"| \'\'\'{if_value(comp_HALE_60, 2)}\'\'\''
        else:
            st += '\n' + '|-\n' + \
                  f'| {prettify_name.get(ser.name, ser.name)} ' + \
                  f'||style="background:#e0ffd8;"| {ser.LE_birth_o:0.2f} ' + \
                  f'||style="background:#eaf3ff;"| {ser.LE_birth_m:0.2f} ' + \
                  f'||style="background:#fee7f6;"| {ser.LE_birth_f:0.2f} ' + \
                  f'|| {if_value(ser.LE_birth_fΔm, 2)} ' + \
                  f'||style="background:#fff8dc;| {if_value(comp_LE_birth, 2)} ' + \
                  f'||style="background:#e0ffd8;border-left-width:2px;"| {ser.HALE_birth_o:0.2f} ' + \
                  f'||style="background:#eaf3ff;"| {ser.HALE_birth_m:0.2f} ' + \
                  f'||style="background:#fee7f6;"| {ser.HALE_birth_f:0.2f} ' + \
                  f'|| {if_value(ser.HALE_birth_fΔm, 2)} ' + \
                  f'||style="background:#fff8dc;| {if_value(comp_HALE_birth, 2)} ' + \
                  f'||style="background:#e0ffd8;border-left-width:3px;"| {ser.LE_60_o:0.2f} ' + \
                  f'||style="background:#eaf3ff;"| {ser.LE_60_m:0.2f} ' + \
                  f'||style="background:#fee7f6;"| {ser.LE_60_f:0.2f} ' + \
                  f'|| {if_value(ser.LE_60_fΔm, 2)} ' + \
                  f'||style="background:#fff8dc;| {if_value(comp_LE_60, 2)} ' + \
                  f'||style="background:#e0ffd8;border-left-width:2px;"| {ser.HALE_60_o:0.2f} ' + \
                  f'||style="background:#eaf3ff;"| {ser.HALE_60_m:0.2f} ' + \
                  f'||style="background:#fee7f6;"| {ser.HALE_60_f:0.2f} ' + \
                  f'|| {if_value(ser.HALE_60_fΔm, 2)} ' + \
                  f'||style="background:#fff8dc;| {if_value(comp_HALE_60, 2)}'
    st += '\n|}'
    
    # WARNING: be sure that inline styles do not contains numbers with comma
    if lang == 'ru':
        st = re.sub('(?<=\\d)\\.(?=\\d)', ',', st)  # replace . to comma, if this . is between two digits
        
    st = st_header + st

    return st

In [56]:
df_regions = df.loc[df.index.isin(LS_WHO_REGIONS+['World'])]
df_regions

,LE_birth_o,LE_birth_m,LE_birth_f,LE_birth_fΔm,LE_birth_Δ_2000,HALE_birth_o,HALE_birth_m,HALE_birth_f,HALE_birth_fΔm,HALE_birth_Δ_2000,ratio_birth_2000,ratio_birth_Δ,ratio_birth,LE_60_o,LE_60_m,LE_60_f,LE_60_fΔm,LE_60_Δ_2000,HALE_60_o,HALE_60_m,HALE_60_f,HALE_60_fΔm,HALE_60_Δ_2000,ratio_60_2000,ratio_60_Δ,ratio_60
,,,,,,,,,,,,,,,,,,,,,,,,,,
Europe,78.10,74.99,81.13,6.14,5.70,67.58,66.07,69.04,2.97,4.69,86.86,-0.33,86.53,22.47,20.48,24.18,3.70,2.91,16.92,15.69,17.98,2.29,2.11,75.72,-0.42,75.30
Western Pacific,77.49,74.51,80.70,6.19,5.53,68.35,66.70,70.13,3.43,4.53,88.69,-0.48,88.20,21.75,19.68,23.84,4.16,2.73,16.64,15.38,17.90,2.52,1.90,77.50,-0.99,76.51
Americas,77.07,74.42,79.76,5.34,2.98,65.76,64.51,67.01,2.50,2.19,85.80,-0.48,85.33,22.64,21.23,23.92,2.69,1.62,16.61,15.74,17.40,1.66,0.96,74.45,-1.09,73.37
World,73.12,70.61,75.70,5.09,6.35,63.45,62.33,64.59,2.26,5.33,87.05,-0.27,86.78,21.03,19.41,22.54,3.13,2.16,15.80,14.87,16.67,1.80,1.52,75.68,-0.54,75.13
South-East Asia,71.37,69.61,73.23,3.62,7.30,61.82,61.38,62.28,0.90,6.48,86.37,0.24,86.62,18.90,17.88,19.89,2.01,1.37,14.03,13.58,14.46,0.88,1.10,73.76,0.47,74.23
Eastern Mediterranean,70.18,68.70,71.75,3.05,4.73,60.52,60.34,60.69,0.35,3.84,86.60,-0.37,86.24,18.88,18.24,19.48,1.24,1.27,14.06,13.85,14.26,0.41,0.78,75.41,-0.94,74.47
Africa,64.17,62.26,66.08,3.82,11.19,55.81,55.14,56.49,1.35,9.83,86.79,0.18,86.97,17.57,16.58,18.44,1.86,2.22,13.29,12.79,13.73,0.94,1.71,75.44,0.20,75.64


In [57]:
table_code_all = create_table_regions(df_regions, lang='en')
output_table_code(table_code_all, 'Table code WHO -regions -en.txt', destination=DESTINATION_OUTPUT)

Data has written to file


In [58]:
table_code_all = create_table_regions(df_regions, lang='ru')
output_table_code(table_code_all, 'Table code WHO -regions -ru.txt', destination=DESTINATION_OUTPUT)

Data has written to file


<br />
<br />
<br />

[regions](https://en.wikipedia.org/wiki/List_of_world_regions_by_life_expectancy) / [регионы](https://ru.wikipedia.org/wiki/Список_регионов_мира_по_ожидаемой_продолжительности_жизни)

In [60]:
df_income_groups = df.loc[df.index.isin(LS_INCOME_GROUPS+['World'])]
df_income_groups

,LE_birth_o,LE_birth_m,LE_birth_f,LE_birth_fΔm,LE_birth_Δ_2000,HALE_birth_o,HALE_birth_m,HALE_birth_f,HALE_birth_fΔm,HALE_birth_Δ_2000,ratio_birth_2000,ratio_birth_Δ,ratio_birth,LE_60_o,LE_60_m,LE_60_f,LE_60_fΔm,LE_60_Δ_2000,HALE_60_o,HALE_60_m,HALE_60_f,HALE_60_fΔm,HALE_60_Δ_2000,ratio_60_2000,ratio_60_Δ,ratio_60
,,,,,,,,,,,,,,,,,,,,,,,,,,
High-income,80.91,78.48,83.33,4.85,3.38,69.28,68.28,70.26,1.98,2.32,86.37,-0.74,85.63,24.28,22.55,25.87,3.32,2.45,18.09,17.02,19.07,2.05,1.63,75.40,-0.90,74.51
Upper-middle-income,75.67,72.73,78.72,5.99,5.87,66.42,64.88,68.01,3.13,4.87,88.18,-0.40,87.78,20.80,18.93,22.57,3.64,2.71,15.82,14.71,16.87,2.16,1.92,76.84,-0.78,76.06
World,73.12,70.61,75.70,5.09,6.35,63.45,62.33,64.59,2.26,5.33,87.05,-0.27,86.78,21.03,19.41,22.54,3.13,2.16,15.80,14.87,16.67,1.80,1.52,75.68,-0.54,75.13
Lower-middle-income,69.57,67.70,71.53,3.83,6.97,60.22,59.70,60.77,1.07,6.14,86.39,0.17,86.56,18.69,17.61,19.73,2.12,1.42,13.91,13.41,14.39,0.98,1.10,74.17,0.25,74.42
Low-income,64.22,62.10,66.34,4.24,10.61,55.89,54.98,56.81,1.83,9.29,86.92,0.10,87.03,17.16,16.14,18.06,1.92,2.40,13.04,12.47,13.53,1.06,1.84,75.88,0.11,75.99


In [61]:
table_code_all = create_table_regions(df_income_groups, lang='en')
output_table_code(table_code_all, 'Table code WHO -income_groups -en.txt', destination=DESTINATION_OUTPUT)

Data has written to file


In [62]:
table_code_all = create_table_regions(df_income_groups, lang='ru')
output_table_code(table_code_all, 'Table code WHO -income_groups -ru.txt', destination=DESTINATION_OUTPUT)

Data has written to file


<br />
<br />
<br />
<hr />

[my private wiki-page for exploration of HALE](https://en.wikipedia.org/wiki/User:Lady3mlnm/HALE)

In [64]:
# create code for placing info about countries in Wikipedia (for countries)
def create_table_countries_extended(df, lang='en'):

    def if_value(x, prec=2):
        return '—' if math.isnan(x) else \
               f"{x:0.{prec}f}"  if x>=0 else \
               f"−{-x:0.{prec}f}"

    def chval(x, prec=2):  # change_value
        return '"| —' if math.isnan(x) else \
               f'color:darkgreen;"| {x:0.{prec}f}' if x>0.0049 else \
               f'color:crimson;"| −{-x:0.{prec}f}' if x<-0.0049 else \
               f'color:darkgray;"| {x:0.{prec}f}'
    
    def chval_bold(x, prec=2):  # change_value
        return '"| —' if math.isnan(x) else \
               f'color:darkgreen;"| \'\'\'{x:0.{prec}f}\'\'\'' if x>0 else \
               f'color:crimson;"| \'\'\'−{-x:0.{prec}f}\'\'\'' if x<0 else \
               f'color:darkgray;"| \'\'\'{x:0.{prec}f}\'\'\''

    if lang=='ru':
        file_header='who_stats_header_countries_extended_ru.txt'
        ptn_1 = 'флагификация'
        ptn_2 = 'флаг'
        prettify_name = {
            "World": "\'\'\'Мир\'\'\'",
            "Europe": "[[Европа]]<ref>{{cite web|title=ВОЗ: Европа |publisher=Всемирная организация здравоохранения |url=https://www.who.int/europe/ru/ |access-date=2025-10-12}}</ref>",
            "Eastern Mediterranean": "\'\'\'{{нп5|Восточное Средиземноморье||en|Eastern Mediterranean}}\'\'\'<ref>{{cite web|title=WHO: Eastern Mediterranean: Countries |lang=en |publisher=Всемирная организация здравоохранения |url=http://www.emro.who.int/countries.html |access-date=2025-10-12}}</ref>",
            "South-East Asia": "\'\'\'[[Юго-Восточная Азия]]\'\'\'<ref>{{cite web|title=WHO: South-East Asia: Where we work |lang=en |publisher=Всемирная организация здравоохранения |url=https://www.who.int/southeastasia/about/where-we-work |access-date=2025-10-12}}</ref>",
            "Western Pacific": "\'\'\'Западно-тихоокеанский регион\'\'\'<ref>{{cite web|title=WHO: Western Pacific: Where we work |lang=en |publisher=Всемирная организация здравоохранения |url=https://www.who.int/westernpacific/about/where-we-work |access-date=2025-10-12}}</ref>",
            "Americas": "\'\'\'[[Америка]]\'\'\'<ref>{{cite web|title=WHO: PAHO: Countries and Centers |lang=en |publisher=Всемирная организация здравоохранения |url=https://www.paho.org/en/countries-and-centers |access-date=2025-10-12}}</ref>",
            "Africa": "\'\'\'[[Африка]]\'\'\'<ref>{{cite web|title=WHO: Africa: Countries |lang=en |publisher=Всемирная организация здравоохранения |url=https://www.afro.who.int/countries |access-date=2025-10-12}}</ref>", 
        }
    else:
        file_header='who_stats_header_countries_extended_en.txt'
        ptn_1 = 'flaglist'
        ptn_2 = 'flagicon'
        prettify_name = {
            "World": "\'\'\'World\'\'\'",
            "Europe": "[[Europe]]<ref>{{cite web|title=WHO: Europe |publisher=World Health Organization |url=https://www.who.int/europe |access-date=12 November 2025}}</ref>",
            "Eastern Mediterranean": "\'\'\'[[Eastern Mediterranean]]\'\'\'<ref>{{cite web|title=WHO: Eastern Mediterranean: Countries |publisher=World Health Organization |url=http://www.emro.who.int/countries.html |access-date=12 November 2025}}</ref>",
            "South-East Asia": "\'\'\'[[South-East Asia]]\'\'\'<ref>{{cite web|title=WHO: South-East Asia: Where we work |language=en |publisher=World Health Organization |url=https://www.who.int/southeastasia/about/where-we-work |access-date=12 November 2025}}</ref>",
            "Western Pacific": "\'\'\'Western Pacific\'\'\'<ref>{{cite web|title=WHO: Western Pacific: Where we work |publisher=World Health Organization |url=https://www.who.int/westernpacific/about/where-we-work |access-date=12 November 2025}}</ref>",
            "Americas": "\'\'\'[[Americas]]\'\'\'<ref>{{cite web|title=WHO: PAHO: Countries and Centers |publisher=World Health Organization |url=https://www.paho.org/en/countries-and-centers |access-date=12 November 2025}}</ref>",
            "Africa": "\'\'\'[[Africa]]\'\'\'<ref>{{cite web|title=WHO: Africa: Countries |language=en |publisher=World Health Organization |url=https://www.afro.who.int/countries |access-date=12 November 2025}}</ref>",
        }
            

    with open('design/' + file_header, mode='r', encoding="utf-8") as fh:
        st_header = fh.read()
        
    st_header = st_header.strip()
    st = ''
        
    for i in range(len(df)):
        ser = df.iloc[i]
        
        comp_LE_birth = ser.LE_birth_Δ_2000
        comp_HALE_birth = ser.HALE_birth_Δ_2000
        comp_LE_60 = ser.LE_60_Δ_2000
        comp_HALE_60 = ser.HALE_60_Δ_2000
        
        if ser.name in ['World', 'Europe', 'Eastern Mediterranean', 'South-East Asia', 'Western Pacific', 'Americas', 'Africa']:
            st += '\n' + '|-class=static-row-header\n' + \
                  f'|style="text-align:center"| {prettify_name.get(ser.name, ser.name)} ' + \
                  f'||style="background:#e0ffd8;"| \'\'\'{ser.LE_birth_o:0.2f}\'\'\' ' + \
                  f'||style="background:#eaf3ff;"| \'\'\'{ser.LE_birth_m:0.2f}\'\'\' ' + \
                  f'||style="background:#fee7f6;"| \'\'\'{ser.LE_birth_f:0.2f}\'\'\' ' + \
                  f'|| \'\'\'{if_value(ser.LE_birth_fΔm, 2)}\'\'\' ' + \
                  f'||style="background:#fff8dc;"| \'\'\'{if_value(comp_LE_birth, 2)}\'\'\' ' + \
                  f'||style="background:#e0ffd8; border-left-width:2px;"| \'\'\'{ser.HALE_birth_o:0.2f}\'\'\' ' + \
                  f'||style="background:#eaf3ff;"| \'\'\'{ser.HALE_birth_m:0.2f}\'\'\' ' + \
                  f'||style="background:#fee7f6;"| \'\'\'{ser.HALE_birth_f:0.2f}\'\'\' ' + \
                  f'|| \'\'\'{if_value(ser.HALE_birth_fΔm, 2)}\'\'\' ' + \
                  f'||style="background:#fff8dc;"| \'\'\'{if_value(comp_HALE_birth, 2)}\'\'\' ' + \
                  f'||style="background:aliceblue;border-left-width:2px;"| \'\'\'{ser.ratio_birth_2000:0.2f}\'\'\' ' + \
                  f'||style="background:aliceblue;{chval_bold(ser.ratio_birth_Δ, 2)} ' + \
                  f'||style="background:aliceblue;"| \'\'\'{ser.ratio_birth:0.2f}\'\'\' ' + \
                  f'||style="background:#e0ffd8; border-left-width:3px;"| \'\'\'{ser.LE_60_o:0.2f}\'\'\' ' + \
                  f'||style="background:#eaf3ff;"| \'\'\'{ser.LE_60_m:0.2f}\'\'\' ' + \
                  f'||style="background:#fee7f6;"| \'\'\'{ser.LE_60_f:0.2f}\'\'\' ' + \
                  f'|| \'\'\'{if_value(ser.LE_60_fΔm, 2)}\'\'\' ' + \
                  f'||style="background:#fff8dc;"| \'\'\'{if_value(comp_LE_60, 2)}\'\'\' ' + \
                  f'||style="background:#e0ffd8; border-left-width:2px;"| \'\'\'{ser.HALE_60_o:0.2f}\'\'\' ' + \
                  f'||style="background:#eaf3ff;"| \'\'\'{ser.HALE_60_m:0.2f}\'\'\' ' + \
                  f'||style="background:#fee7f6;"| \'\'\'{ser.HALE_60_f:0.2f}\'\'\' ' + \
                  f'|| \'\'\'{if_value(ser.HALE_60_fΔm, 2)}\'\'\' ' + \
                  f'||style="background:#fff8dc;"| \'\'\'{if_value(comp_HALE_60, 2)}\'\'\' ' + \
                  f'||style="background:aliceblue;border-left-width:2px;"| \'\'\'{ser.ratio_60_2000:0.2f}\'\'\' ' + \
                  f'||style="background:aliceblue;{chval_bold(ser.ratio_60_Δ, 2)} ' + \
                  f'||style="background:aliceblue;"| \'\'\'{ser.ratio_60:0.2f}\'\'\' ' + \
                  f'||'
        else:
            st += '\n' + '|-\n' + \
                  f'| {{{{{ptn_1}|{ser.name}}}}} ' + \
                  f'||style="background:#e0ffd8;"| {ser.LE_birth_o:0.2f} ' + \
                  f'||style="background:#eaf3ff;"| {ser.LE_birth_m:0.2f} ' + \
                  f'||style="background:#fee7f6;"| {ser.LE_birth_f:0.2f} ' + \
                  f'|| {if_value(ser.LE_birth_fΔm, 2)} ' + \
                  f'||style="background:#fff8dc;| {if_value(comp_LE_birth, 2)} ' + \
                  f'||style="background:#e0ffd8;border-left-width:2px;"| {ser.HALE_birth_o:0.2f} ' + \
                  f'||style="background:#eaf3ff;"| {ser.HALE_birth_m:0.2f} ' + \
                  f'||style="background:#fee7f6;"| {ser.HALE_birth_f:0.2f} ' + \
                  f'|| {if_value(ser.HALE_birth_fΔm, 2)} ' + \
                  f'||style="background:#fff8dc;| {if_value(comp_HALE_birth, 2)} ' + \
                  f'||style="background:aliceblue;border-left-width:2px;"| {ser.ratio_birth_2000:0.2f} ' + \
                  f'||style="background:aliceblue;{chval(ser.ratio_birth_Δ, 2)} ' + \
                  f'||style="background:aliceblue;"| {ser.ratio_birth:0.2f} ' + \
                  f'||style="background:#e0ffd8;border-left-width:3px;"| {ser.LE_60_o:0.2f} ' + \
                  f'||style="background:#eaf3ff;"| {ser.LE_60_m:0.2f} ' + \
                  f'||style="background:#fee7f6;"| {ser.LE_60_f:0.2f} ' + \
                  f'|| {if_value(ser.LE_60_fΔm, 2)} ' + \
                  f'||style="background:#fff8dc;| {if_value(comp_LE_60, 2)} ' + \
                  f'||style="background:#e0ffd8;border-left-width:2px;"| {ser.HALE_60_o:0.2f} ' + \
                  f'||style="background:#eaf3ff;"| {ser.HALE_60_m:0.2f} ' + \
                  f'||style="background:#fee7f6;"| {ser.HALE_60_f:0.2f} ' + \
                  f'|| {if_value(ser.HALE_60_fΔm, 2)} ' + \
                  f'||style="background:#fff8dc;| {if_value(comp_HALE_60, 2)} ' + \
                  f'||style="background:aliceblue;border-left-width:2px;"| {ser.ratio_60_2000:0.2f} ' + \
                  f'||style="background:aliceblue;{chval(ser.ratio_60_Δ, 2)} ' + \
                  f'||style="background:aliceblue;"| {ser.ratio_60:0.2f} ' + \
                  f'|| {{{{{ptn_2}|{ser.name}}}}}'
    st += '\n|}'
    # f'||style="background:#fff8dc;{chval(comp_LE_birth, 2)} ' + \
    
    # WARNING: be sure that inline styles do not contains numbers with comma
    if lang == 'ru':
        st = re.sub('(?<=\\d)\\.(?=\\d)', ',', st)  # replace . to comma, if this . is between two digits
        
    st = st_header + st

    return st

In [65]:
table_code_all = create_table_countries_extended(df_all_countries, lang='en')
output_table_code(table_code_all, 'Table code WHO -all_countries -extended -en.txt', destination=DESTINATION_OUTPUT)

Data has written to file


<br />
<br />

In [67]:
# create code for placing info about countries in Wikipedia (for countries)
def create_table_regions_extended(df, lang='en'):

    def if_value(x, prec=2):
        return '—' if math.isnan(x) else \
               f"{x:0.{prec}f}"  if x>=0 else \
               f"−{-x:0.{prec}f}"

    def chval(x, prec=2):  # change_value
        return '"| —' if math.isnan(x) else \
               f'color:darkgreen;"| {x:0.{prec}f}' if x>0.0049 else \
               f'color:crimson;"| −{-x:0.{prec}f}' if x<-0.0049 else \
               f'color:darkgray;"| {x:0.{prec}f}'
    
    def chval_bold(x, prec=2):  # change_value
        return '"| —' if math.isnan(x) else \
               f'color:darkgreen;"| \'\'\'{x:0.{prec}f}\'\'\'' if x>0 else \
               f'color:crimson;"| \'\'\'−{-x:0.{prec}f}\'\'\'' if x<0 else \
               f'color:darkgray;"| \'\'\'{x:0.{prec}f}\'\'\''

    if lang=='ru':
        file_header='who_stats_header_regions_extended_ru.txt'
        prettify_name = {
            "World": "\'\'\'Мир\'\'\'",
            "Europe": "[[Европа]]<ref>{{cite web|title=ВОЗ: Европа |publisher=Всемирная организация здравоохранения |url=https://www.who.int/europe/ru/ |access-date=2025-10-12}}</ref>",
            "Eastern Mediterranean": "Восточное Средиземноморье<ref>{{cite web|title=WHO: Eastern Mediterranean: Countries |lang=en |publisher=Всемирная организация здравоохранения |url=http://www.emro.who.int/countries.html |access-date=2025-10-12}}</ref>",
            "South-East Asia": "[[Юго-Восточная Азия]]<ref>{{cite web|title=WHO: South-East Asia: Where we work |lang=en |publisher=Всемирная организация здравоохранения |url=https://www.who.int/southeastasia/about/where-we-work |access-date=2025-10-12}}</ref>",
            "Western Pacific": "Западно-тихоокеанский регион<ref>{{cite web|title=WHO: Western Pacific: Where we work |lang=en |publisher=Всемирная организация здравоохранения |url=https://www.who.int/westernpacific/about/where-we-work |access-date=2025-10-12}}</ref>",
            "Americas": "[[Америка]]<ref>{{cite web|title=WHO: PAHO: Countries and Centers |lang=en |publisher=Всемирная организация здравоохранения |url=https://www.paho.org/en/countries-and-centers |access-date=2025-10-12}}</ref>",
            "Africa": "[[Африка]]<ref>{{cite web|title=WHO: Africa: Countries |lang=en |publisher=Всемирная организация здравоохранения |url=https://www.afro.who.int/countries |access-date=2025-10-12}}</ref>", 
        }
    else:
        file_header='who_stats_header_regions_extended_en.txt'
        prettify_name = {
            "World": "\'\'\'World\'\'\'",
            "Europe": "[[Europe]]<ref>{{cite web|title=WHO: Europe |publisher=World Health Organization |url=https://www.who.int/europe |access-date=12 November 2025}}</ref>",
            "Eastern Mediterranean": "[[Eastern Mediterranean]]<ref>{{cite web|title=WHO: Eastern Mediterranean: Countries |publisher=World Health Organization |url=http://www.emro.who.int/countries.html |access-date=12 November 2025}}</ref>",
            "South-East Asia": "[[South-East Asia]]<ref>{{cite web|title=WHO: South-East Asia: Where we work |language=en |publisher=World Health Organization |url=https://www.who.int/southeastasia/about/where-we-work |access-date=12 November 2025}}</ref>",
            "Western Pacific": "Western Pacific<ref>{{cite web|title=WHO: Western Pacific: Where we work |publisher=World Health Organization |url=https://www.who.int/westernpacific/about/where-we-work |access-date=12 November 2025}}</ref>",
            "Americas": "[[Americas]]<ref>{{cite web|title=WHO: PAHO: Countries and Centers |publisher=World Health Organization |url=https://www.paho.org/en/countries-and-centers |access-date=12 November 2025}}</ref>",
            "Africa": "[[Africa]]<ref>{{cite web|title=WHO: Africa: Countries |language=en |publisher=World Health Organization |url=https://www.afro.who.int/countries |access-date=12 November 2025}}</ref>",
        } 
            

    with open('design/' + file_header, mode='r', encoding="utf-8") as fh:
        st_header = fh.read()
        
    st_header = st_header.strip()
    st = ''
        
    for i in range(len(df)):
        ser = df.iloc[i]
        
        comp_LE_birth = ser.LE_birth_Δ_2000
        comp_HALE_birth = ser.HALE_birth_Δ_2000
        comp_LE_60 = ser.LE_60_Δ_2000
        comp_HALE_60 = ser.HALE_60_Δ_2000
        
        if ser.name in ['World']:
            st += '\n' + '|-class=static-row-header\n' + \
                  f'|style="text-align:center"| {prettify_name.get(ser.name, ser.name)} ' + \
                  f'||style="background:#e0ffd8;"| \'\'\'{ser.LE_birth_o:0.2f}\'\'\' ' + \
                  f'||style="background:#eaf3ff;"| \'\'\'{ser.LE_birth_m:0.2f}\'\'\' ' + \
                  f'||style="background:#fee7f6;"| \'\'\'{ser.LE_birth_f:0.2f}\'\'\' ' + \
                  f'|| \'\'\'{if_value(ser.LE_birth_fΔm, 2)}\'\'\' ' + \
                  f'||style="background:#fff8dc;"| \'\'\'{if_value(comp_LE_birth, 2)}\'\'\' ' + \
                  f'||style="background:#e0ffd8; border-left-width:2px;"| \'\'\'{ser.HALE_birth_o:0.2f}\'\'\' ' + \
                  f'||style="background:#eaf3ff;"| \'\'\'{ser.HALE_birth_m:0.2f}\'\'\' ' + \
                  f'||style="background:#fee7f6;"| \'\'\'{ser.HALE_birth_f:0.2f}\'\'\' ' + \
                  f'|| \'\'\'{if_value(ser.HALE_birth_fΔm, 2)}\'\'\' ' + \
                  f'||style="background:#fff8dc;"| \'\'\'{if_value(comp_HALE_birth, 2)}\'\'\' ' + \
                  f'||style="background:aliceblue;border-left-width:2px;"| \'\'\'{ser.ratio_birth_2000:0.2f}\'\'\' ' + \
                  f'||style="background:aliceblue;{chval_bold(ser.ratio_birth_Δ, 2)} ' + \
                  f'||style="background:aliceblue;"| \'\'\'{ser.ratio_birth:0.2f}\'\'\' ' + \
                  f'||style="background:#e0ffd8; border-left-width:3px;"| \'\'\'{ser.LE_60_o:0.2f}\'\'\' ' + \
                  f'||style="background:#eaf3ff;"| \'\'\'{ser.LE_60_m:0.2f}\'\'\' ' + \
                  f'||style="background:#fee7f6;"| \'\'\'{ser.LE_60_f:0.2f}\'\'\' ' + \
                  f'|| \'\'\'{if_value(ser.LE_60_fΔm, 2)}\'\'\' ' + \
                  f'||style="background:#fff8dc;"| \'\'\'{if_value(comp_LE_60, 2)}\'\'\' ' + \
                  f'||style="background:#e0ffd8; border-left-width:2px;"| \'\'\'{ser.HALE_60_o:0.2f}\'\'\' ' + \
                  f'||style="background:#eaf3ff;"| \'\'\'{ser.HALE_60_m:0.2f}\'\'\' ' + \
                  f'||style="background:#fee7f6;"| \'\'\'{ser.HALE_60_f:0.2f}\'\'\' ' + \
                  f'|| \'\'\'{if_value(ser.HALE_60_fΔm, 2)}\'\'\' ' + \
                  f'||style="background:#fff8dc;"| \'\'\'{if_value(comp_HALE_60, 2)}\'\'\' ' + \
                  f'||style="background:aliceblue;border-left-width:2px;"| \'\'\'{ser.ratio_60_2000:0.2f}\'\'\' ' + \
                  f'||style="background:aliceblue;{chval_bold(ser.ratio_60_Δ, 2)} ' + \
                  f'||style="background:aliceblue;"| \'\'\'{ser.ratio_60:0.2f}\'\'\''
        else:
            st += '\n' + '|-\n' + \
                  f'| {prettify_name.get(ser.name, ser.name)} ' + \
                  f'||style="background:#e0ffd8;"| {ser.LE_birth_o:0.2f} ' + \
                  f'||style="background:#eaf3ff;"| {ser.LE_birth_m:0.2f} ' + \
                  f'||style="background:#fee7f6;"| {ser.LE_birth_f:0.2f} ' + \
                  f'|| {if_value(ser.LE_birth_fΔm, 2)} ' + \
                  f'||style="background:#fff8dc;| {if_value(comp_LE_birth, 2)} ' + \
                  f'||style="background:#e0ffd8;border-left-width:2px;"| {ser.HALE_birth_o:0.2f} ' + \
                  f'||style="background:#eaf3ff;"| {ser.HALE_birth_m:0.2f} ' + \
                  f'||style="background:#fee7f6;"| {ser.HALE_birth_f:0.2f} ' + \
                  f'|| {if_value(ser.HALE_birth_fΔm, 2)} ' + \
                  f'||style="background:#fff8dc;| {if_value(comp_HALE_birth, 2)} ' + \
                  f'||style="background:aliceblue;border-left-width:2px;"| {ser.ratio_birth_2000:0.2f} ' + \
                  f'||style="background:aliceblue;{chval(ser.ratio_birth_Δ, 2)} ' + \
                  f'||style="background:aliceblue;"| {ser.ratio_birth:0.2f} ' + \
                  f'||style="background:#e0ffd8;border-left-width:3px;"| {ser.LE_60_o:0.2f} ' + \
                  f'||style="background:#eaf3ff;"| {ser.LE_60_m:0.2f} ' + \
                  f'||style="background:#fee7f6;"| {ser.LE_60_f:0.2f} ' + \
                  f'|| {if_value(ser.LE_60_fΔm, 2)} ' + \
                  f'||style="background:#fff8dc;| {if_value(comp_LE_60, 2)} ' + \
                  f'||style="background:#e0ffd8;border-left-width:2px;"| {ser.HALE_60_o:0.2f} ' + \
                  f'||style="background:#eaf3ff;"| {ser.HALE_60_m:0.2f} ' + \
                  f'||style="background:#fee7f6;"| {ser.HALE_60_f:0.2f} ' + \
                  f'|| {if_value(ser.HALE_60_fΔm, 2)} ' + \
                  f'||style="background:#fff8dc;| {if_value(comp_HALE_60, 2)} ' + \
                  f'||style="background:aliceblue;border-left-width:2px;"| {ser.ratio_60_2000:0.2f} ' + \
                  f'||style="background:aliceblue;{chval(ser.ratio_60_Δ, 2)} ' + \
                  f'||style="background:aliceblue;"| {ser.ratio_60:0.2f}'
    st += '\n|}'
    
    # WARNING: be sure that inline styles do not contains numbers with comma
    if lang == 'ru':
        st = re.sub('(?<=\\d)\\.(?=\\d)', ',', st)  # replace . to comma, if this . is between two digits
        
    st = st_header + st

    return st

In [68]:
# df_regions = df.loc[df.index.isin(LS_WHO_REGIONS+['World'])]
# df_regions

In [69]:
table_code_all = create_table_regions_extended(df_regions, lang='en')
output_table_code(table_code_all, 'Table code WHO -regions -extended -en.txt', destination=DESTINATION_OUTPUT)

Data has written to file


<br />
<br />

In [71]:
# df_income_groups = df.loc[df.index.isin(LS_INCOME_GROUPS+['World'])]
# df_income_groups

In [72]:
table_code_all = create_table_regions_extended(df_income_groups, lang='en')
output_table_code(table_code_all, 'Table code WHO -income_groups -extended -en.txt', destination=DESTINATION_OUTPUT)

Data has written to file


<br />
<br />

In [74]:
# check that all countries are considered for local tables
ls_all_countries = sorted(df.index.to_list())
print(len(ls_all_countries))

196


In [75]:
# combine all local lists into one

import itertools
all_local_lists = \
    sorted(set(itertools.chain(ls_Europe, ls_Asia, ls_Oceania, ls_N_America, ls_S_America, ls_Africa,
                                             LS_WHO_REGIONS, LS_INCOME_GROUPS, LS_IGNORE)))
print(len(all_local_lists))

196


In [76]:
[country for country in ls_all_countries if country not in all_local_lists]

[]

In [77]:
[country for country in all_local_lists if country not in ls_all_countries]

[]

<br />
<br />

In [79]:
# play beep to denote completion of the program
import IPython.display as ipd
import numpy as np

# manually generated sound
t = 1  # time is seconds
beep = np.sin(2*np.pi*400*np.arange(10000*t)/10000)
ipd.Audio(beep, rate=10000, autoplay=True)